# ConformalLadder — Stage 1, end to end (Kaggle edition)

**Layer-resolved, distribution-free error budgets and FDR-controlled routing for electronic-structure calculations.**

This notebook rebuilds the whole Stage-1 study from scratch:

| step | data | what is produced |
|---|---|---|
| Pilot 0 | QM9-G4MP2 (130 258 molecules; B3LYP/6-31G(2df,p) → G4MP2) | conformal intervals, FDR-controlled certification, behaviour under shift |
| Pilot 1 | MultiXC-QM9 (76 functionals × SZ/DZP/TZP) + G4MP2 — 21 468 isomerization energies | four-layer ladder (basis / functional / dispersion / remainder) + non-additivity |
| Pilot 2 | QM7b multilevel (HF/MP2/CCSD(T) × STO-3G/6-31G/cc-pVDZ, 7 211 molecules) | correlation × basis non-additivity of composite schemes, certified |
| Gates | shift scenarios of pilots 1–2 | abstention gates + escalate-to-recalibrate loop |
| W4-17 | own PySCF grid: 8 functionals × {def2-SVP, def2-TZVP} + D3/D4 + FOD on 215 species | layer budget per rung, W4 references |
| Tables | everything above | `results/tables/*.csv` + `SUMMARY.md`, all figures, one zip |

**Before running**
1. *Settings → Internet → On* (Figshare / GitHub downloads, ~0.7 GB).
2. Optional but recommended — attach a private Kaggle dataset containing
   * `ct8b00832_si_001.zip` — the QM7b Supporting Information of Zaspel *et al.*, *JCTC* 2019, **15**, 1546 (ACS blocks scripted downloads; download it once by hand from the article page). Kaggle extracts uploaded archives automatically — that is fine, the extracted `supplementary/` folder is detected too.
   * `w4-17_grid.jsonl` — a precomputed W4-17 grid (skips the 2–3 h PySCF campaign).
   * optionally `molecules_qm9.csv` (or the `molecules_qm9.partNN.csv.gz` parts) if Figshare is slow.
3. Choose the settings in the next cell. A CPU session is enough (4 cores); total runtime ≈ 30 min without the campaign, ≈ 3 h with it.

Runtime layout: `/kaggle/working/ConformalLadder/{src,notebooks,data,results}`; the final zip is `/kaggle/working/ConformalLadder_results.zip`.


In [ ]:
# ------------------------------------------------------------------ settings
RUN_W4_CAMPAIGN   = True      # PySCF functional x basis grid on W4-17 (~2-3 h on 4 cores); skipped if a precomputed w4-17_grid.jsonl is attached
W4_FUNCTIONALS    = "PBE,B3LYP,PBE0,TPSS,r2SCAN,M06-2X,BLYP,wB97X"   # e.g. "PBE,B3LYP,PBE0" for a ~40-min campaign
RUN_RECALIBRATION = True      # escalate-to-recalibrate experiment (~5-8 min)
CLEAN_RAW_CSV     = True      # delete the 533 MB MultiXC csv after conversion to parquet (keeps the notebook output small)

import os, sys, subprocess, shutil, time, pathlib, json
ON_KAGGLE = pathlib.Path("/kaggle/working").exists()
CL_ROOT = os.environ.get("CL_ROOT_OVERRIDE") or ("/kaggle/working/ConformalLadder" if ON_KAGGLE else str(pathlib.Path.cwd() / "ConformalLadder"))
ROOT = pathlib.Path(CL_ROOT)
for d in ["src/conformalladder", "notebooks", "data/raw", "data/processed", "results", "logs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
NT = os.cpu_count() or 2
ENV = dict(os.environ, PYTHONPATH=str(ROOT / "src"), OMP_NUM_THREADS="1", MPLBACKEND="Agg", CL_THREADS=str(NT), CL_W4_FUNCTIONALS=W4_FUNCTIONALS)

def run(cmd, threads=1, tail=60, quiet=("kekulize", "PerformanceWarning", "Warning", "warn", "out[")):
    """Run a project script, stream a filtered log, raise on failure."""
    env = dict(ENV, OMP_NUM_THREADS=str(threads), CL_FIG_NO_SUPTITLE="1"); t0 = time.time()
    if cmd.startswith("python "):
        cmd = sys.executable + cmd[6:]
    p = subprocess.Popen(cmd, shell=True, cwd=str(ROOT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        if not any(q in line for q in quiet):
            lines.append(line.rstrip())
    p.wait()
    print("\n".join(lines[-tail:]))
    print(f"[{'ok' if p.returncode == 0 else 'FAILED'} in {time.time()-t0:.0f}s] {cmd}")
    if p.returncode != 0:
        raise RuntimeError(cmd)

from IPython.display import Image, display
def show(rel, width=1100):
    display(Image(filename=str(ROOT / rel), width=width))
print("project root:", ROOT, "| cores:", NT, "| Kaggle:", ON_KAGGLE)

## 1. Software
PySCF (DFT), dftd3/dftd4 (dispersion), RDKit (bond-type features), pyarrow, tabulate. Everything else is in the Kaggle image.

In [ ]:
import importlib, subprocess, sys
need = [p for p, m in [("pyscf", "pyscf"), ("dftd3", "dftd3"), ("dftd4", "dftd4"), ("rdkit", "rdkit"), ("pyarrow", "pyarrow"), ("tabulate", "tabulate"), ("scipy", "scipy"), ("scikit-learn", "sklearn")]
        if importlib.util.find_spec(m) is None]
if need:
    print("installing", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import pyscf, sklearn, rdkit; print("pyscf", pyscf.__version__, "| sklearn", sklearn.__version__, "| rdkit", rdkit.__version__)

## 2. Project code
The library (`src/conformalladder`) and the analysis scripts (`notebooks/`) are written to disk exactly as used in the study.

In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/__init__.py
"""ConformalLadder: layer-resolved conformal error budgets and FDR-controlled routing
for electronic-structure calculations."""

__version__ = "0.0.1"


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/conformal.py
"""Distribution-free machinery used throughout ConformalLadder.

All functions are deliberately small and explicit so that every guarantee can be
traced to a textbook statement:

* split conformal with a locally scaled score          (Lei et al. 2018)
* conformalized quantile regression, CQR               (Romano, Patterson & Candès 2019)
* weighted conformal under covariate shift             (Tibshirani et al. 2019)
* conformal p-values + Benjamini–Hochberg selection    (Jin & Candès 2023)
* group-conditional (Mondrian) coverage diagnostics    (Vovk 2012)
"""
from __future__ import annotations

import numpy as np


# --------------------------------------------------------------------------- #
# Finite-sample quantile helpers
# --------------------------------------------------------------------------- #
def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    """The ceil((n+1)(1-alpha))/n empirical quantile of calibration scores.

    Using this quantile guarantees P(Y in C(X)) >= 1 - alpha for an exchangeable
    test point (marginal coverage)."""
    s = np.sort(np.asarray(scores, dtype=float))
    n = s.size
    k = int(np.ceil((n + 1) * (1 - alpha)))
    if k > n:
        return np.inf
    return float(s[k - 1])


def weighted_conformal_quantile(scores: np.ndarray, weights_cal: np.ndarray,
                                weight_test: float, alpha: float) -> float:
    """Quantile of the weighted empirical distribution used in weighted conformal
    prediction (Tibshirani et al. 2019).  `weights_cal` are likelihood-ratio
    estimates w(x_j) = dP_test(x_j)/dP_cal(x_j) for the calibration points and
    `weight_test` is w(x_test).  Returns +inf when the test point is so far out of
    the calibration support that no finite quantile is guaranteed."""
    s = np.asarray(scores, dtype=float)
    w = np.asarray(weights_cal, dtype=float)
    order = np.argsort(s)
    s, w = s[order], w[order]
    p = w / (w.sum() + weight_test)
    cdf = np.cumsum(p)
    idx = np.searchsorted(cdf, 1 - alpha, side="left")
    if idx >= s.size:
        return np.inf
    return float(s[idx])


# --------------------------------------------------------------------------- #
# Interval constructions
# --------------------------------------------------------------------------- #
def scaled_residual_intervals(mu_cal, sigma_cal, y_cal, mu_test, sigma_test,
                              alpha: float):
    """Locally scaled split conformal: score |y - mu(x)| / sigma(x).

    Returns lower, upper arrays for the test points and the calibrated quantile."""
    sigma_cal = np.maximum(np.asarray(sigma_cal, float), 1e-6)
    sigma_test = np.maximum(np.asarray(sigma_test, float), 1e-6)
    scores = np.abs(np.asarray(y_cal) - np.asarray(mu_cal)) / sigma_cal
    q = conformal_quantile(scores, alpha)
    return mu_test - q * sigma_test, mu_test + q * sigma_test, q


def cqr_intervals(qlo_cal, qhi_cal, y_cal, qlo_test, qhi_test, alpha: float):
    """Conformalized quantile regression (Romano et al. 2019)."""
    y_cal = np.asarray(y_cal, float)
    scores = np.maximum(np.asarray(qlo_cal) - y_cal, y_cal - np.asarray(qhi_cal))
    q = conformal_quantile(scores, alpha)
    return np.asarray(qlo_test) - q, np.asarray(qhi_test) + q, q


def weighted_scaled_intervals(mu_cal, sigma_cal, y_cal, w_cal,
                              mu_test, sigma_test, w_test, alpha: float):
    """Weighted conformal (covariate shift) version of the scaled-residual interval.
    One quantile per test point, hence O(n_cal * n_test)."""
    sigma_cal = np.maximum(np.asarray(sigma_cal, float), 1e-6)
    sigma_test = np.maximum(np.asarray(sigma_test, float), 1e-6)
    scores = np.abs(np.asarray(y_cal) - np.asarray(mu_cal)) / sigma_cal
    order = np.argsort(scores)
    s_sorted = scores[order]
    w_sorted = np.asarray(w_cal, float)[order]
    cum_w = np.cumsum(w_sorted)
    total_w = cum_w[-1]
    q = np.empty(len(mu_test))
    for i, wt in enumerate(np.asarray(w_test, float)):
        target = (1 - alpha) * (total_w + wt)
        idx = np.searchsorted(cum_w, target, side="left")
        q[i] = np.inf if idx >= s_sorted.size else s_sorted[idx]
    return mu_test - q * sigma_test, mu_test + q * sigma_test, q


# --------------------------------------------------------------------------- #
# Selection with FDR control (Jin & Candès 2023)
# --------------------------------------------------------------------------- #
def conformal_pvalues_small_error(abs_err_cal, conf_cal, conf_test,
                                  threshold: float, rng=None):
    """Conformal p-values for the null H_i : |err_i| >= threshold  ("molecule i is NOT
    within `threshold` of the reference").  Rejecting H_i certifies the cheap
    (corrected) value; BH on these p-values controls the FDR of the certified set.

    Construction (Jin & Candès, JMLR 2023, "clipped" score).  Let Y' = -|err| and
    c' = -threshold, so the null is Y' <= c'.  With a confidence statistic conf(x) >= 0
    (smaller = more confident, e.g. the predicted |residual| or an interval half-width)
    define the monotone score
        V(x, y') = M            if y' > c'   (calibration molecule within threshold)
                 = conf(x)      if y' <= c'  (calibration molecule outside threshold)
    Test scores are evaluated at the null boundary: V(x_i, c') = conf(x_i).  Hence
        p_i = ( #{ j : |err_j| >= threshold and conf_j < conf_i } + U (1 + ties) ) / (n + 1),
    i.e. the rank of the test molecule's confidence among the calibration molecules
    that *failed*.  Under H_i the oracle score equals the test score, so p_i is exactly
    uniform under exchangeability (no conservativeness is lost)."""
    rng = np.random.default_rng(0) if rng is None else rng
    bad = np.asarray(abs_err_cal, float) >= threshold
    v_bad_sorted = np.sort(np.asarray(conf_cal, float)[bad])
    n = np.asarray(abs_err_cal).size
    v_test = np.asarray(conf_test, float)
    less = np.searchsorted(v_bad_sorted, v_test, side="left")
    leq = np.searchsorted(v_bad_sorted, v_test, side="right")
    ties = leq - less
    u = rng.uniform(size=v_test.size)
    return (less + u * (1 + ties)) / (n + 1)


def benjamini_hochberg(pvals: np.ndarray, q: float) -> np.ndarray:
    """Boolean mask of rejections at FDR level q."""
    p = np.asarray(pvals, float)
    m = p.size
    order = np.argsort(p)
    thresh = q * (np.arange(1, m + 1)) / m
    passed = p[order] <= thresh
    if not passed.any():
        return np.zeros(m, dtype=bool)
    k = np.max(np.nonzero(passed)[0])
    rej = np.zeros(m, dtype=bool)
    rej[order[: k + 1]] = True
    return rej


# --------------------------------------------------------------------------- #
# Diagnostics
# --------------------------------------------------------------------------- #
def coverage(y, lo, hi) -> float:
    y = np.asarray(y, float)
    return float(np.mean((y >= lo) & (y <= hi)))


def group_coverage(y, lo, hi, groups) -> dict:
    """Coverage and mean width per group (Mondrian / conditional diagnostic)."""
    y = np.asarray(y, float)
    groups = np.asarray(groups)
    out = {}
    for g in np.unique(groups):
        m = groups == g
        finite = np.isfinite(hi[m]) & np.isfinite(lo[m])
        out[str(g)] = {
            "n": int(m.sum()),
            "coverage": float(np.mean((y[m] >= lo[m]) & (y[m] <= hi[m]))),
            "mean_width": float(np.mean((hi[m] - lo[m])[finite])) if finite.any() else float("inf"),
        }
    return out


def selection_report(selected, abs_err_true, threshold: float) -> dict:
    """Empirical false-discovery proportion and power of a certified set."""
    selected = np.asarray(selected, bool)
    good = np.asarray(abs_err_true, float) < threshold
    n_sel = int(selected.sum())
    fdp = float(np.mean(~good[selected])) if n_sel else 0.0
    power = float(selected[good].mean()) if good.any() else 0.0
    return {"n_test": int(selected.size), "n_selected": n_sel,
            "fraction_selected": n_sel / selected.size,
            "FDP": fdp, "power": power,
            "base_rate_good": float(good.mean())}


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/layers.py
"""Layer-resolved error budgets from a functional × basis grid on benchmark reactions.

Definitions (all in kcal/mol, per reaction r with reference R_r):

    total error          eps  = R  - dE(xc_c, b_c)                      (cheap level c)
    basis layer          beta = dE(xc_c, b_big) - dE(xc_c, b_c)
    functional layer     phi  = dE(xc_*, b_big) - dE(xc_c, b_big)         (xc_* = "best" functional)
    dispersion layer     delta= dE_disp(xc_*)                             (D3(BJ) contribution to dE)
    correlation/remainder rho = R - dE(xc_*, b_big) - delta               (beyond-DFT + beyond-b_big)

    eps = beta + phi + delta + rho   holds exactly (telescoping identity).

Non-additivity is the path dependence of the corrections, i.e. the factorial interaction

    I_bf = [dE(xc_*, b_big) - dE(xc_c, b_big)] - [dE(xc_*, b_c) - dE(xc_c, b_c)]

(the functional correction measured in the large basis minus the same correction measured
in the small basis).  A purely additive composite scheme assumes I_bf = 0.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
HARTREE_TO_KCAL = 627.509474
DB = ROOT / "data" / "raw" / "ACCDB" / "Databases"


# --------------------------------------------------------------------------- #
# Reference reactions
# --------------------------------------------------------------------------- #
def load_reactions(setname="W4-17", include_re=False) -> pd.DataFrame:
    """ACCDB DatasetEval format: id, (coef, species)*, reference[kcal/mol]."""
    files = [DB / setname / f"DatasetEval.{setname}_kcal.csv"]
    if include_re:
        files.append(DB / setname / f"DatasetEval.{setname}-RE_kcal.csv")
    rows = []
    for f in files:
        for line in f.read_text().split("\n"):
            if not line.strip():
                continue
            parts = line.split(",")
            rid, ref = parts[0], float(parts[-1])
            terms = parts[1:-1]
            stoich = {}
            for c, s in zip(terms[0::2], terms[1::2]):
                stoich[s] = stoich.get(s, 0.0) + float(c)
            subset = rid.rsplit("_", 1)[0]
            rows.append({"rid": rid, "subset": subset, "stoich": stoich, "ref": ref})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- #
# Grid records
# --------------------------------------------------------------------------- #
def load_grid(path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    recs = [json.loads(l) for l in Path(path).read_text().split("\n") if l.strip()]
    df = pd.DataFrame(recs)
    disp = df[df["basis"] == "none"].copy()
    fod = df[df["xc"] == "FOD_TPSS_T5000"].copy()
    dft = df[(df["basis"] != "none") & (df["xc"] != "FOD_TPSS_T5000")].copy()
    return dft, disp, fod


def reaction_energies(reactions: pd.DataFrame, dft: pd.DataFrame, disp: pd.DataFrame,
                      require_converged=True) -> pd.DataFrame:
    """Wide table: one row per reaction, one column per (xc|basis) and per dispersion tag."""
    d = dft[dft["converged"]] if require_converged else dft
    E = {(r.species, r.xc, r.basis): r.E for r in d.itertuples()}
    Ed = {(r.species, r.xc): r.E for r in disp.itertuples() if r.E is not None}
    combos = sorted({(x, b) for (_, x, b) in E})
    dtags = sorted({x for (_, x) in Ed})
    out = []
    for r in reactions.itertuples():
        row = {"rid": r.rid, "subset": r.subset, "ref": r.ref, "n_species": len(r.stoich),
               "sum_abs_coef": float(sum(abs(c) for c in r.stoich.values()))}
        for x, b in combos:
            try:
                row[f"{x}|{b}"] = HARTREE_TO_KCAL * sum(c * E[(s, x, b)] for s, c in r.stoich.items())
            except KeyError:
                row[f"{x}|{b}"] = np.nan
        for t in dtags:
            try:
                row[f"disp:{t}"] = HARTREE_TO_KCAL * sum(c * Ed[(s, t)] for s, c in r.stoich.items())
            except KeyError:
                row[f"disp:{t}"] = np.nan
        out.append(row)
    return pd.DataFrame(out)


COVALENT_RADII = {"H": 0.31, "He": 0.28, "Li": 1.28, "Be": 0.96, "B": 0.84, "C": 0.76, "N": 0.71, "O": 0.66,
                  "F": 0.57, "Ne": 0.58, "Na": 1.66, "Mg": 1.41, "Al": 1.21, "Si": 1.11, "P": 1.07, "S": 1.05,
                  "Cl": 1.02, "Ar": 1.06, "Br": 1.20, "I": 1.39}
_ELEMENT_ORDER = ["H", "B", "C", "N", "O", "F", "Al", "Si", "P", "S", "Cl", "Br", "I", "Be", "Li", "Na", "Mg"]


def geometry_bond_counts(symbols, coords) -> dict:
    """Connectivity from covalent radii: counts of bonds per element pair, split into
    'single-like' (d > 0.92 * (r_A + r_B)) and 'short' (multiple-bond-like) contacts."""
    out = {}
    n = len(symbols)
    for i in range(n):
        for j in range(i + 1, n):
            ra, rb = COVALENT_RADII.get(symbols[i], 1.2), COVALENT_RADII.get(symbols[j], 1.2)
            d = float(np.linalg.norm(coords[i] - coords[j]))
            if d < 1.25 * (ra + rb):
                a, b = sorted([symbols[i], symbols[j]], key=lambda s: _ELEMENT_ORDER.index(s) if s in _ELEMENT_ORDER else 99)
                tag = "short" if d < 0.92 * (ra + rb) else "single"
                out[f"bond_{a}{b}_{tag}"] = out.get(f"bond_{a}{b}_{tag}", 0) + 1
    out["n_bonds"] = sum(out.values())
    return out


def species_features(reactions: pd.DataFrame, dft: pd.DataFrame, fod: pd.DataFrame,
                     xc_c: str, b_c: str, geom_dir=None) -> pd.DataFrame:
    """Cheap per-reaction features aggregated from the species that enter the reaction,
    using only the cheap level (xc_c, b_c), the FOD run and the geometry (bond counts)."""
    d = dft[(dft["xc"] == xc_c) & (dft["basis"] == b_c)].set_index("species")
    f = fod.set_index("species") if len(fod) else None
    geom_dir = Path(geom_dir) if geom_dir else ROOT / "data" / "raw" / "ACCDB" / "Geometries"
    bond_cache, elem_cache = {}, {}
    from conformalladder.compute_grid import load_xyz
    def bonds(s):
        if s not in bond_cache:
            sy, co, _, _ = load_xyz(geom_dir / f"{s}.xyz")
            bond_cache[s] = geometry_bond_counts(sy, co)
            elem_cache[s] = {e: sy.count(e) for e in set(sy)}
        return bond_cache[s], elem_cache[s]
    rows = []
    for r in reactions.itertuples():
        sp = list(r.stoich.keys()); w = np.array([abs(c) for c in r.stoich.values()])
        net_b, gross_b, gross_e = {}, {}, {}
        for s, c in r.stoich.items():
            bc, ec = bonds(s)
            for k, v in bc.items():
                net_b[k] = net_b.get(k, 0.0) + c * v; gross_b[k] = gross_b.get(k, 0.0) + abs(c) * v
            for e, v in ec.items():
                gross_e[f"elem_{e}"] = gross_e.get(f"elem_{e}", 0.0) + abs(c) * v
        def agg(col, how, src=d):
            vals = np.array([src[col].get(s, np.nan) if s in src.index else np.nan for s in sp], float)
            if np.all(np.isnan(vals)):
                return np.nan
            if how == "min": return np.nanmin(vals)
            if how == "max": return np.nanmax(vals)
            if how == "wsum": return float(np.nansum(vals * w))
            return float(np.nanmean(vals))
        row = {"rid": r.rid,
               "gap_min": agg("gap", "min"), "gap_mean": agg("gap", "mean"),
               "s2_wsum": agg("s2", "wsum"), "s2_max": agg("s2", "max"),
               "dipole_max": agg("dipole", "max"),
               "nelec_wsum": agg("n_elec", "wsum"), "natom_wsum": agg("natom", "wsum"),
               "nbf_max": agg("nbf", "max"), "n_open_shell": float(sum(d["spin"].get(s, 0) > 0 for s in sp if s in d.index)),
               "n_species": len(sp), "sum_abs_coef": float(w.sum())}
        if f is not None:
            row["fod_wsum"] = agg("n_fod", "wsum", f); row["fod_max"] = agg("n_fod", "max", f)
        row.update({f"net_{k}": v for k, v in net_b.items()})
        row.update({f"gross_{k}": v for k, v in gross_b.items()})
        row.update(gross_e)
        rows.append(row)
    return pd.DataFrame(rows).fillna(0.0)


# --------------------------------------------------------------------------- #
# Budget
# --------------------------------------------------------------------------- #
def layer_budget(re: pd.DataFrame, xc_c: str, b_c: str, xc_best: str, b_big: str,
                 disp_tag: str = "D3BJ") -> pd.DataFrame:
    """Per-reaction exact decomposition eps = beta + phi + delta + rho and the interaction I_bf."""
    c = re[f"{xc_c}|{b_c}"]; c_big = re[f"{xc_c}|{b_big}"]
    s_small = re[f"{xc_best}|{b_c}"]; s_big = re[f"{xc_best}|{b_big}"]
    delta = re.get(f"disp:{xc_best}+{disp_tag}", pd.Series(np.nan, index=re.index))
    delta_c = re.get(f"disp:{xc_c}+{disp_tag}", pd.Series(np.nan, index=re.index))
    out = pd.DataFrame({"rid": re["rid"], "subset": re["subset"], "ref": re["ref"],
                        "dE_cheap": c,
                        "eps_total": re["ref"] - c,
                        "beta_basis": c_big - c,
                        "phi_functional": s_big - c_big,
                        "delta_dispersion": delta,
                        "rho_remainder": re["ref"] - s_big - delta,
                        "I_bf": (s_big - c_big) - (s_small - c),
                        "I_fd": delta - delta_c,            # dispersion correction depends on the functional
                        "eps_after_ladder": re["ref"] - s_big - delta})
    out["check_identity"] = out["eps_total"] - (out["beta_basis"] + out["phi_functional"] + out["delta_dispersion"] + out["rho_remainder"])
    return out


def summarize_budget(b: pd.DataFrame) -> pd.DataFrame:
    cols = ["eps_total", "beta_basis", "phi_functional", "delta_dispersion", "rho_remainder", "I_bf", "I_fd", "eps_after_ladder"]
    rows = []
    for sub, g in list(b.groupby("subset")) + [("ALL", b)]:
        row = {"subset": sub, "n": int(g[cols[0]].notna().sum())}
        for cname in cols:
            v = g[cname].dropna()
            row[f"{cname}_MAE"] = float(np.mean(np.abs(v))) if len(v) else np.nan
            row[f"{cname}_mean"] = float(v.mean()) if len(v) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/gates.py
"""Abstention gates for the router: decide, from covariates only, whether a conformal interval /
certificate computed from the calibration set can be trusted for a new item.

Three gates (all return a boolean mask `answer` for the test items, True = answer, False = abstain):

* weighted_conformal_gate  — the weighted-conformal quantile is infinite when the estimated
  likelihood ratio of the test point exceeds alpha/(1-alpha) times the calibration weight mass
  (Tibshirani et al. 2019).  Exact when weights are known; with estimated weights it abstains
  whenever the test item is *separable* from the calibration set, even if the score distribution
  is unaffected (too conservative for benign shifts — pilots 1 and 2).

* knn_support_gate — abstain when the test item lies outside the support of the calibration
  covariates: its distance to the k-th nearest calibration neighbour (standardised features)
  exceeds the (1 - tail) quantile of the calibration leave-one-out k-NN distances.

* conditional_coverage_gate — abstain when a cross-fitted classifier, trained on the calibration
  set to predict the miscoverage event {score > q_hat} from x, predicts a miscoverage probability
  larger than `factor * alpha` at the test item.  This targets what actually matters (does the
  nonconformity score depend on the direction of the shift?) but can only be trusted inside the
  calibration support, so it is normally combined with the k-NN gate.
"""
from __future__ import annotations

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def knn_support_gate(X_cal, X_test, k: int = 10, tail: float = 0.01):
    sc = StandardScaler().fit(X_cal)
    A, B = sc.transform(X_cal), sc.transform(X_test)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(A)
    d_cal = nn.kneighbors(A, return_distance=True)[0][:, k]        # leave-one-out: k-th neighbour excluding self
    d_test = nn.kneighbors(B, n_neighbors=k, return_distance=True)[0][:, k - 1]
    thr = np.quantile(d_cal, 1 - tail)
    return d_test <= thr, d_test / thr


def conditional_coverage_gate(X_cal, scores_cal, q_hat, X_test, alpha: float, factor: float = 2.0, seed: int = 0):
    miss = (np.asarray(scores_cal) > q_hat).astype(int)
    if miss.sum() < 5 or miss.sum() > len(miss) - 5:
        return np.ones(len(X_test), bool), np.full(len(X_test), miss.mean())
    clf = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40,
                                         l2_regularization=1.0, random_state=seed)
    clf.fit(X_cal, miss)
    g = clf.predict_proba(X_test)[:, 1]
    return g <= factor * alpha, g


def weighted_conformal_gate(X_cal, X_test, alpha: float, seed: int = 0):
    """Abstain iff the weighted-conformal quantile would be infinite (estimated weights, cross-fitted)."""
    Xs = np.vstack([X_cal, X_test]); lab = np.r_[np.zeros(len(X_cal)), np.ones(len(X_test))]
    clf = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_leaf_nodes=15, min_samples_leaf=40, random_state=seed)
    p = cross_val_predict(clf, Xs, lab, cv=5, method="predict_proba")[:, 1]
    prior = len(X_test) / len(Xs)
    with np.errstate(divide="ignore", invalid="ignore"):
        w = (p / (1 - p)) * ((1 - prior) / prior)
    w = np.where(p >= 1 - 1e-12, np.inf, w)
    w_cal = np.clip(w[: len(X_cal)], 1e-8, 1e8); w_test = w[len(X_cal):]
    p_test = w_test / (w_cal.sum() + w_test)
    return p_test <= alpha, w_test


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/data_qm9g4mp2.py
"""Tidy table for the QM9-G4MP2 pilot.

Source: Ward et al., MRS Commun. 2019 (github.com/globus-labs/g4mp2-atomization-energy).
Each of 130,258 QM9 molecules carries B3LYP/6-31G(2df,p) properties (from QM9) and
G4MP2 energies on the same geometry.

Target (kcal/mol):
    eps = De_at(G4MP2) - De_at(B3LYP)
where De_at is the *electronic* atomization energy (ZPE removed at each level:
QM9 uses the raw harmonic B3LYP ZPE, G4MP2 the same ZPE scaled by 0.9854).
Sign convention: atomization energies are negative in the source, so eps > 0
means B3LYP over-binds relative to G4MP2.

Features are restricted to quantities that come *for free* with the cheap
calculation (orbital energies, dipole, polarizability, ZPE, composition) plus
bond-type counts from the SMILES, so that the pilot mimics the deployment
situation: cheap calculation done, expensive one not yet.
"""
from __future__ import annotations

from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

HARTREE_TO_KCAL = 627.509474

RAW = Path(__file__).resolve().parents[2] / "data" / "raw" / "g4mp2-atomization-energy" / "data" / "output" / "g4mp2_data.json.gz"
OUT = Path(__file__).resolve().parents[2] / "data" / "processed" / "qm9_g4mp2_pilot.parquet"

ELEMENTS = ["C", "H", "N", "O", "F"]
BOND_KEYS = ["C-C", "C=C", "C#C", "C:C", "C-H", "C-N", "C=N", "C#N", "C:N",
             "C-O", "C=O", "C:O", "C-F", "N-H", "N-N", "N=N", "N:N", "N-O", "N=O",
             "O-H", "O-O", "N:O"]


def _bond_counts(smiles: str) -> dict:
    from rdkit import Chem
    from rdkit.Chem import rdMolDescriptors
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {}
    mol = Chem.AddHs(mol)
    sym = {1.0: "-", 2.0: "=", 3.0: "#", 1.5: ":"}
    rank = {"C": 0, "N": 1, "O": 2, "F": 3, "H": 4}   # heavy atoms first -> "C-H", "N-H", "O-H", "C-N", ...
    c = Counter()
    for b in mol.GetBonds():
        a1, a2 = b.GetBeginAtom().GetSymbol(), b.GetEndAtom().GetSymbol()
        order = sym.get(b.GetBondTypeAsDouble(), "~")
        e1, e2 = sorted([a1, a2], key=lambda s: rank.get(s, 99))
        c[f"{e1}{order}{e2}"] += 1
    c["n_rings"] = rdMolDescriptors.CalcNumRings(mol)
    c["n_arom_rings"] = rdMolDescriptors.CalcNumAromaticRings(mol)
    c["n_rot_bonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)
    return dict(c)


def build(force: bool = False) -> pd.DataFrame:
    if OUT.exists() and not force:
        return pd.read_parquet(OUT)
    df = pd.read_json(RAW, lines=True)

    # composition from the xyz block
    comp = []
    for xyz in df["xyz"]:
        lines = xyz.split("\n")[2:]
        comp.append(Counter(l.split()[0] for l in lines if l.strip()))
    for el in ELEMENTS:
        df[f"n_{el}"] = [c.get(el, 0) for c in comp]

    # electronic atomization energies (Hartree, negative = bound)
    df["De_b3lyp"] = df["u0_atom"] - df["zpe"]
    df["De_g4mp2"] = df["g4mp2_atom"] - df["g4mp2_zpe"]
    df["eps_kcal"] = (df["De_g4mp2"] - df["De_b3lyp"]) * HARTREE_TO_KCAL
    df["De_g4mp2_kcal"] = df["De_g4mp2"] * HARTREE_TO_KCAL
    df["De_b3lyp_kcal"] = df["De_b3lyp"] * HARTREE_TO_KCAL

    # bond-type counts (prefer SMILES re-derived from the relaxed geometry)
    bc = [_bond_counts(s1 if isinstance(s1, str) and s1 else s0)
          for s0, s1 in zip(df["smiles_0"], df["smiles_1"])]
    bc = pd.DataFrame(bc).fillna(0)
    for k in BOND_KEYS + ["n_rings", "n_arom_rings", "n_rot_bonds"]:
        df[k] = bc[k].values if k in bc else 0

    keep = (["index", "filename", "smiles_0", "smiles_1", "n_atom", "n_heavy_atoms",
             "n_electrons", "homo", "lumo", "bandgap", "mu", "alpha", "R2", "zpe",
             "cv", "A", "B", "C", "in_holdout"]
            + [f"n_{el}" for el in ELEMENTS] + BOND_KEYS
            + ["n_rings", "n_arom_rings", "n_rot_bonds",
               "De_b3lyp_kcal", "De_g4mp2_kcal", "eps_kcal"])
    out = df[keep].copy()
    OUT.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(OUT, index=False)
    return out


FEATURES = (["n_atom", "n_heavy_atoms", "n_electrons", "homo", "lumo", "bandgap", "mu",
             "alpha", "R2", "zpe", "cv"] + [f"n_{el}" for el in ELEMENTS] + BOND_KEYS
            + ["n_rings", "n_arom_rings", "n_rot_bonds"])

if __name__ == "__main__":
    d = build(force=True)
    print(d.shape)
    print(d["eps_kcal"].describe())


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/data_multixc.py
"""MultiXC-QM9 (Nandi, Vegge & Bhowmik 2023) merged with QM9-G4MP2 (Ward et al. 2019).

molecules_qm9.csv: electronic atomization energies (eV, negative = bound) for 76 functionals x
{SZ, DZP, TZP} Slater bases (ADF; hybrids/meta-GGAs evaluated post-SCF) + GFN2-xTB, on the QM9
B3LYP/6-31G(2df,p) geometries.  QM9-G4MP2 supplies the reference: De(G4MP2) = D0 - scaled ZPE.

Everything is converted to kcal/mol with the sign convention  AE < 0  (bound).
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
EV_TO_KCAL = 23.060548
OUT = ROOT / "data" / "processed" / "multixc_merged.parquet"
BASES = ["SZ", "DZP", "TZP"]


def functionals(df: pd.DataFrame):
    return sorted({c[:-4] for c in df.columns if c.endswith("_TZP")})


def build(force: bool = False) -> pd.DataFrame:
    if OUT.exists() and not force:
        return pd.read_parquet(OUT)
    mx = pd.read_parquet(ROOT / "data" / "processed" / "multixc_all_bases.parquet")
    mx["index"] = mx["index"].astype(int)
    from conformalladder.data_qm9g4mp2 import build as build_g4
    g4 = build_g4()
    g4 = g4.rename(columns={"index": "index"})
    g4["index"] = g4["index"].astype(int)
    energy_cols = [c for c in mx.columns if c.rsplit("_", 1)[-1] in BASES] + ["GFNXTB"]
    mx[energy_cols] = mx[energy_cols].astype(float) * EV_TO_KCAL
    merged = g4.merge(mx[["index", "chemformula"] + energy_cols], on="index", how="inner")
    merged.to_parquet(OUT, index=False)
    return merged


if __name__ == "__main__":
    m = build(force=True)
    print(m.shape)
    print("functionals:", len(functionals(m)))
    r = m[m["index"] == 1].iloc[0]
    print("CH4: De(G4MP2) %.2f | PBE_TZP %.2f | B3LYP(VWN5)_TZP %.2f | M06-2X_TZP %.2f kcal/mol" % (r["De_g4mp2_kcal"], r["PBE_TZP"], r["B3LYP(VWN5)_TZP"], r["M06-2X_TZP"]))


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/data_qm7b.py
"""QM7b multilevel set (Zaspel, Huang, Harbrecht & von Lilienfeld, JCTC 2019, 15, 1546; ACS SI).

7,211 molecules (H, C, N, O, S, Cl; <= 7 heavy atoms), total energies at
{HF, MP2, CCSD(T)} x {STO-3G, 6-31G, cc-pVDZ} on identical geometries, plus ground-state atomic
energies at the same nine levels.  We build *true* electronic atomization energies
AE = E_mol - sum_I n_I E_atom,I  (kcal/mol, negative = bound) for every level.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
SI = ROOT / "data" / "raw" / "qm7b_multilevel" / "supplementary"
OUT = ROOT / "data" / "processed" / "qm7b_multilevel.parquet"
HARTREE_TO_KCAL = 627.509474
METHODS = ["HF", "MP2", "CCSD(T)"]
BASES = {"sto3g": "STO-3G", "631g": "6-31G", "ccpvdz": "cc-pVDZ"}
Z_ORDER = [1, 6, 7, 8, 16, 17]
SYM = {1: "H", 6: "C", 7: "N", 8: "O", 16: "S", 17: "Cl"}


def _atoms(basis_key):
    rows = {}
    for line in (SI / "atoms" / f"E_atom_{basis_key}.txt").read_text().split("\n"):
        if not line.strip() or line.startswith("#"):
            continue
        z, *e = line.split()
        rows[int(z)] = np.array([float(x) for x in e[:3]]) * HARTREE_TO_KCAL
    return rows


def build(force: bool = False) -> pd.DataFrame:
    if OUT.exists() and not force:
        return pd.read_parquet(OUT)
    nzs = np.loadtxt(SI / "nzs.dat", comments="#", dtype=int)
    n = nzs.shape[0]
    df = pd.DataFrame({"mol": np.arange(1, n + 1)})
    for z, sym in SYM.items():
        df[f"n_{sym}"] = nzs[:, Z_ORDER.index(z)]
    df["n_heavy"] = df[[f"n_{s}" for s in ["C", "N", "O", "S", "Cl"]]].sum(axis=1)
    df["n_atom"] = nzs.sum(axis=1)
    for bkey, bname in BASES.items():
        etot = np.loadtxt(SI / f"Etot_{bkey}.txt")
        at = _atoms(bkey)
        for j, meth in enumerate(METHODS):
            atom_sum = sum(nzs[:, Z_ORDER.index(z)] * at[z][j] for z in Z_ORDER)
            df[f"Etot|{meth}|{bname}"] = etot[:, j]
            df[f"AE|{meth}|{bname}"] = etot[:, j] - atom_sum
    df["Enn"] = np.loadtxt(SI / "Enn.dat")
    # geometry-derived bond counts (cheap, geometry only)
    from conformalladder.layers import geometry_bond_counts
    bc = []
    for i in range(1, n + 1):
        lines = (SI / "geometry" / f"frag_{i:04d}.xyz").read_text().strip().split("\n")
        na = int(lines[0]); atoms = [l.split() for l in lines[2:2 + na]]
        sy = [a[0] for a in atoms]; co = np.array([[float(x) for x in a[1:4]] for a in atoms])
        bc.append(geometry_bond_counts(sy, co))
    bc = pd.DataFrame(bc).fillna(0.0)
    bc.columns = [f"bc_{c}" for c in bc.columns]
    df = pd.concat([df, bc], axis=1)
    # formula string for isomer grouping
    df["formula"] = ["".join(f"{s}{int(r[f'n_{s}'])}" for s in ["C", "H", "N", "O", "S", "Cl"] if r[f"n_{s}"] > 0) for _, r in df.iterrows()]
    df.to_parquet(OUT, index=False)
    return df


if __name__ == "__main__":
    d = build(force=True)
    print(d.shape)
    r = d.iloc[0]
    print("mol 1 (CH4): AE CCSD(T)/cc-pVDZ = %.1f ; HF/cc-pVDZ = %.1f ; CCSD(T)/6-31G = %.1f kcal/mol" % (r["AE|CCSD(T)|cc-pVDZ"], r["AE|HF|cc-pVDZ"], r["AE|CCSD(T)|6-31G"]))
    print("formulas:", d["formula"].nunique(), "| elements S/Cl present:", int(((d.n_S > 0) | (d.n_Cl > 0)).sum()))


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/compute_grid.py
"""Compute a functional × basis grid (plus dispersion and cheap diagnostics) on a list of
ACCDB species with PySCF.  Results are appended to a JSONL file, one record per
(species, xc, basis), so the campaign is resumable and partial grids are usable.

Usage:
    PYTHONPATH=src python -m conformalladder.compute_grid --set W4-17 [--out data/processed/w4-17_grid.jsonl]

Records:
    {"species", "xc", "basis", "E", "converged", "gap", "s2", "dipole", "homo", "lumo",
     "n_elec", "spin", "charge", "natom", "nbf", "seconds"}
    dispersion records use xc = "<functional>+D3BJ" / "+D3ZERO" / "+D4" and basis = "none"
    FOD records use xc = "FOD_TPSS_T5000", basis = "def2-TZVP", field "n_fod"
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[2]
GEOM = ROOT / "data" / "raw" / "ACCDB" / "Geometries"
LISTS = ROOT / "data" / "raw" / "ACCDB" / "Databases"

# (label used in results, PySCF xc string)
FUNCTIONALS = [("PBE", "PBE"), ("B3LYP", "B3LYP"), ("PBE0", "PBE0"), ("TPSS", "TPSS"),
               ("r2SCAN", "R2SCAN"), ("M06-2X", "M06-2X"), ("BLYP", "BLYP"), ("wB97X", "WB97X")]
BASES = ["def2-SVP", "def2-TZVP"]
BOHR = 0.52917721092
KT_5000K = 5000.0 * 3.166811563e-6   # Hartree


def load_xyz(path: Path):
    lines = path.read_text().strip().split("\n")
    n = int(lines[0])
    charge, mult = map(int, lines[1].split()[:2])
    atoms = [l.split() for l in lines[2:2 + n]]
    symbols = [a[0] for a in atoms]
    coords = np.array([[float(x) for x in a[1:4]] for a in atoms])
    return symbols, coords, charge, mult


def species_list(setname: str):
    lst = LISTS / setname / f"{setname}_ALL.list"
    return [l.strip() for l in lst.read_text().split("\n") if l.strip()]


def run_dft(symbols, coords, charge, mult, xc, basis, smear_kT=None):
    from pyscf import dft, gto, lib
    from pyscf.scf import addons
    lib.num_threads(int(os.environ.get("CL_THREADS", os.cpu_count() or 2)))
    atom = [(s, tuple(c)) for s, c in zip(symbols, coords)]
    mol = gto.M(atom=atom, charge=charge, spin=mult - 1, basis=basis, verbose=0, max_memory=3500)
    mf = dft.UKS(mol) if mult > 1 else dft.RKS(mol)
    mf.xc = xc
    mf = mf.density_fit()
    mf.conv_tol = 1e-8
    mf.max_cycle = 120
    if smear_kT:
        mf = addons.smearing_(mf, sigma=smear_kT, method="fermi")
    t0 = time.time()
    e = mf.kernel()
    if not mf.converged and not smear_kT:
        mf2 = mf.newton()
        mf2.max_cycle = 60
        e = mf2.kernel(mf.make_rdm1())
        mf = mf2 if mf2.converged else mf
        e = mf.e_tot
    rec = {"E": float(e), "converged": bool(mf.converged), "natom": mol.natm, "nbf": int(mol.nao),
           "n_elec": int(mol.nelectron), "charge": charge, "spin": mult - 1, "seconds": round(time.time() - t0, 1)}
    try:
        occ = np.asarray(mf.mo_occ); ene = np.asarray(mf.mo_energy)
        if occ.ndim == 2:   # UKS
            homo = max(ene[s][occ[s] > 0.5].max() for s in range(2) if (occ[s] > 0.5).any())
            lumo_c = [ene[s][occ[s] < 0.5].min() for s in range(2) if (occ[s] < 0.5).any()]
            lumo = min(lumo_c) if lumo_c else float("nan")
            rec["s2"] = float(mf.spin_square()[0])
        else:
            homo = float(ene[occ > 1.0].max()); lumo = float(ene[occ < 1.0].min()) if (occ < 1.0).any() else float("nan")
            rec["s2"] = 0.0
        rec["homo"], rec["lumo"], rec["gap"] = float(homo), float(lumo), float(lumo - homo)
        rec["dipole"] = float(np.linalg.norm(mf.dip_moment(unit="Debye", verbose=0)))
        if smear_kT:
            # N_FOD = electrons displaced from the aufbau (T = 0) occupations
            if occ.ndim == 2:
                nfod = 0.0
                for s in range(2):
                    nocc = int(round(mol.nelec[s])); order = np.argsort(ene[s]); f = occ[s][order]
                    theta = np.zeros_like(f); theta[:nocc] = 1.0
                    nfod += float(np.abs(theta - f).sum() / 2)
            else:
                nocc = mol.nelectron // 2; order = np.argsort(ene); f = occ[order]
                theta = np.zeros_like(f); theta[:nocc] = 2.0
                nfod = float(np.abs(theta - f).sum() / 2)
            rec["n_fod"] = nfod
    except Exception as exc:  # diagnostics are optional
        rec["diag_error"] = str(exc)[:200]
    return rec


def run_dispersion(symbols, coords, label):
    """D3(BJ), D3(zero) and D4 energies (Hartree) for a functional label; missing parameters -> None."""
    out = {}
    from pyscf.data.elements import charge as elem_charge
    numbers = np.array([elem_charge(s) for s in symbols]); pos = coords / BOHR
    key = label.lower().replace("-", "")
    try:
        from dftd3.interface import DispersionModel, RationalDampingParam, ZeroDampingParam
        m = DispersionModel(numbers, pos)
        for tag, P in (("D3BJ", RationalDampingParam), ("D3ZERO", ZeroDampingParam)):
            try:
                out[tag] = float(m.get_dispersion(P(method=key), grad=False)["energy"])
            except Exception:
                out[tag] = None
    except Exception as exc:
        out["d3_error"] = str(exc)[:120]
    try:
        from dftd4.interface import DampingParam, DispersionModel as D4
        try:
            out["D4"] = float(D4(numbers, pos).get_dispersion(DampingParam(method=key), grad=False)["energy"])
        except Exception:
            out["D4"] = None
    except Exception as exc:
        out["d4_error"] = str(exc)[:120]
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--set", default="W4-17")
    ap.add_argument("--out", default=None)
    ap.add_argument("--only-dispersion", action="store_true")
    args = ap.parse_args()
    out = Path(args.out) if args.out else ROOT / "data" / "processed" / f"{args.set.lower()}_grid.jsonl"
    out.parent.mkdir(parents=True, exist_ok=True)
    done = set()
    if out.exists():
        for line in out.read_text().split("\n"):
            if line.strip():
                r = json.loads(line); done.add((r["species"], r["xc"], r["basis"]))
    species = species_list(args.set)
    geoms = {s: load_xyz(GEOM / f"{s}.xyz") for s in species}
    log = open(out.with_suffix(".log"), "a")

    def emit(rec):
        with open(out, "a") as f:
            f.write(json.dumps(rec) + "\n")
        done.add((rec["species"], rec["xc"], rec["basis"]))

    # 1) dispersion (seconds for the whole set)
    for s in species:
        sy, co, q, m = geoms[s]
        for label, _ in FUNCTIONALS:
            if (s, f"{label}+D3BJ", "none") in done:
                continue
            d = run_dispersion(sy, co, label)
            for tag in ("D3BJ", "D3ZERO", "D4"):
                emit({"species": s, "xc": f"{label}+{tag}", "basis": "none", "E": d.get(tag),
                      "converged": d.get(tag) is not None, "natom": len(sy)})
    if args.only_dispersion:
        return

    # 2) DFT grid, ordered so complete (xc, basis) blocks appear early
    order = [("PBE", "def2-SVP"), ("B3LYP", "def2-SVP"), ("PBE", "def2-TZVP"), ("B3LYP", "def2-TZVP"),
             ("PBE0", "def2-SVP"), ("PBE0", "def2-TZVP"), ("FOD", "def2-TZVP"),
             ("TPSS", "def2-SVP"), ("TPSS", "def2-TZVP"), ("r2SCAN", "def2-SVP"), ("r2SCAN", "def2-TZVP"),
             ("M06-2X", "def2-SVP"), ("M06-2X", "def2-TZVP"), ("BLYP", "def2-SVP"), ("BLYP", "def2-TZVP"),
             ("wB97X", "def2-SVP"), ("wB97X", "def2-TZVP")]
    xc_map = dict(FUNCTIONALS)
    wanted = os.environ.get("CL_W4_FUNCTIONALS")
    if wanted:
        keep = {w.strip() for w in wanted.split(",")} | {"FOD"}
        order = [(l, b) for l, b in order if l in keep]
    for label, basis in order:
        t_block = time.time(); n_new = 0
        for s in species:
            key = ("FOD_TPSS_T5000" if label == "FOD" else label)
            if (s, key, basis) in done:
                continue
            sy, co, q, m = geoms[s]
            try:
                if label == "FOD":
                    rec = run_dft(sy, co, q, m, "TPSS", basis, smear_kT=KT_5000K)
                else:
                    rec = run_dft(sy, co, q, m, xc_map[label], basis)
            except Exception as exc:
                rec = {"E": None, "converged": False, "error": str(exc)[:300]}
            rec.update({"species": s, "xc": key, "basis": basis})
            emit(rec); n_new += 1
        log.write(f"{time.strftime('%H:%M:%S')} block {label}/{basis}: {n_new} new records in {time.time()-t_block:.0f}s\n"); log.flush()
    log.write("ALL DONE\n"); log.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/kaggle_data.py
"""Data acquisition for a fresh environment (Kaggle / Colab / any Linux box with internet).

Fetches exactly what the Stage-1 pipeline needs and lays it out in the project tree:

    data/raw/g4mp2-atomization-energy/data/output/g4mp2_data.json.gz   QM9-G4MP2 (Ward et al. 2019)   ~78 MB
    data/raw/multixc_qm9/molecules_qm9.csv                              MultiXC-QM9 (Nandi et al. 2023) 533 MB
    data/raw/ACCDB/{Databases,Geometries}                               ACCDB (Morgante & Peverati 2019)  ~60 MB
    data/raw/qm7b_multilevel/supplementary/...                          QM7b multilevel (Zaspel et al. 2019, ACS SI)

The ACS Supporting Information server refuses scripted downloads; if the direct request fails the
file is looked for in the attached Kaggle datasets (/kaggle/input/**/ct8b00832_si_001.zip).
A precomputed W4-17 grid (w4-17_grid.jsonl) attached the same way lets the notebook skip the
2-3 h PySCF campaign.

Usage:  python -m conformalladder.kaggle_data  [--root PATH] [--skip-multixc]
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
UA = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) ConformalLadder/1.0"}
INPUT_DIRS = [p for p in ("/kaggle/input", "/content/drive/MyDrive", os.environ.get("CL_INPUT_DIR", "")) if p]

FIGSHARE_MULTIXC_CSV = ("https://ndownloader.figshare.com/files/37216291", 532561703)   # molecules_qm9.csv
G4MP2_RAW = "https://raw.githubusercontent.com/globus-labs/g4mp2-atomization-energy/master/data/output/g4mp2_data.json.gz"
G4MP2_ZIP = "https://codeload.github.com/globus-labs/g4mp2-atomization-energy/zip/refs/heads/master"
ACCDB_ZIP = "https://codeload.github.com/peverati/ACCDB/zip/refs/heads/master"
QM7B_SI = "https://pubs.acs.org/doi/suppl/10.1021/acs.jctc.8b00832/suppl_file/ct8b00832_si_001.zip"


def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)


def find_attached(pattern: str, want_dir: bool = False):
    """First file (or directory) matching `pattern` in the attached input directories (Kaggle datasets etc.).
    NB: Kaggle extracts every uploaded .zip automatically, so an attached archive usually appears as a folder."""
    for d in INPUT_DIRS:
        hits = sorted(glob.glob(os.path.join(d, "**", pattern), recursive=True))
        hits = [h for h in hits if (os.path.isdir(h) if want_dir else os.path.isfile(h))]
        if hits:
            return Path(hits[0])
    return None


def download(url: str, dest: Path, size: int | None = None, tries: int = 5) -> bool:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and (size is None or dest.stat().st_size == size):
        log(f"= {dest.name} present ({dest.stat().st_size/1e6:.1f} MB)"); return True
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(tries):
        have = part.stat().st_size if part.exists() else 0
        headers = dict(UA)
        if have:
            headers["Range"] = f"bytes={have}-"
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=180) as r:
                if have and r.getcode() != 206:
                    have = 0
                total = have + int(r.headers.get("Content-Length") or 0) if r.headers.get("Content-Length") else size
                done, t0, last = have, time.time(), 0.0
                with open(part, "ab" if have else "wb") as f:
                    while True:
                        chunk = r.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk); done += len(chunk)
                        if time.time() - last > 5:
                            pct = f"{100*done/total:5.1f}%" if total else ""
                            print(f"\r  {dest.name}: {done/1e6:8.1f} MB {pct}  {(done-have)/max(time.time()-t0,1e-6)/1e6:5.1f} MB/s", end="", flush=True)
                            last = time.time()
                print()
            if size is not None and part.stat().st_size != size:
                raise IOError(f"size mismatch {part.stat().st_size} != {size}")
            part.replace(dest)
            return True
        except Exception as exc:
            log(f"! attempt {attempt+1}/{tries} for {dest.name}: {exc}")
            if any(code in str(exc) for code in ("403", "404")):     # refused / missing: retrying will not help
                return False
            if "416" in str(exc) and part.exists():
                part.unlink()
            time.sleep(3 * (attempt + 1))
    return False


def unzip(src: Path, dest_dir: Path):
    with zipfile.ZipFile(src) as zf:
        zf.extractall(dest_dir)


# --------------------------------------------------------------------------- datasets
def fetch_qm9_g4mp2(root: Path) -> Path:
    dest = root / "data" / "raw" / "g4mp2-atomization-energy" / "data" / "output" / "g4mp2_data.json.gz"
    if dest.exists():
        log("= QM9-G4MP2 present"); return dest
    att = find_attached("g4mp2_data.json.gz")
    if att:
        dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(att, dest); log(f"copied attached {att}"); return dest
    if download(G4MP2_RAW, dest):
        return dest
    log("raw.githubusercontent failed -> repository zip")
    z = root / "data" / "raw" / "g4mp2.zip"
    if download(G4MP2_ZIP, z):
        unzip(z, root / "data" / "raw"); z.unlink()
        src = root / "data" / "raw" / "g4mp2-atomization-energy-master"
        if src.exists():
            shutil.move(str(src), str(root / "data" / "raw" / "g4mp2-atomization-energy"))
        return dest
    log("zip failed -> git clone")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/globus-labs/g4mp2-atomization-energy.git",
                    str(root / "data" / "raw" / "g4mp2-atomization-energy")], check=True)
    return dest


def fetch_accdb(root: Path) -> Path:
    dest = root / "data" / "raw" / "ACCDB"
    if (dest / "Geometries").exists():
        log("= ACCDB present"); return dest
    z = root / "data" / "raw" / "accdb.zip"
    if download(ACCDB_ZIP, z):
        unzip(z, root / "data" / "raw"); z.unlink()
        shutil.move(str(root / "data" / "raw" / "ACCDB-master"), str(dest))
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/peverati/ACCDB.git", str(dest)], check=True)
    return dest


def fetch_multixc(root: Path) -> Path:
    """molecules_qm9.csv (76 functionals x SZ/DZP/TZP + GFN2-xTB, eV) -> data/processed/multixc_all_bases.parquet"""
    parquet = root / "data" / "processed" / "multixc_all_bases.parquet"
    if parquet.exists():
        log("= MultiXC-QM9 parquet present"); return parquet
    csv = root / "data" / "raw" / "multixc_qm9" / "molecules_qm9.csv"
    att = find_attached("molecules_qm9.csv")
    if att and not csv.exists():
        csv.parent.mkdir(parents=True, exist_ok=True); shutil.copy(att, csv); log(f"copied attached {att}")
    if not csv.exists():
        packed = find_attached("molecules_qm9.part01.csv.gz") or find_attached("molecules_qm9.part01.csv")   # packed parts (gz or auto-extracted)
        if packed:
            import pandas as pd
            parts = sorted(glob.glob(os.path.join(packed.parent, "molecules_qm9.part*.csv*")))
            log(f"reading {len(parts)} packed parts")
            df = pd.concat([pd.read_csv(p, dtype={"index": str}) for p in parts], ignore_index=True)
            parquet.parent.mkdir(parents=True, exist_ok=True); df.to_parquet(parquet, index=False); return parquet
        url, size = FIGSHARE_MULTIXC_CSV
        if not download(url, csv, size):
            raise RuntimeError("MultiXC-QM9 download failed (Figshare) and no attached copy found")
    import pandas as pd
    log("converting molecules_qm9.csv -> parquet")
    df = pd.read_csv(csv, dtype={"index": str})
    parquet.parent.mkdir(parents=True, exist_ok=True); df.to_parquet(parquet, index=False)
    return parquet


def fetch_qm7b(root: Path) -> Path:
    """QM7b multilevel SI.  Accepts, in this order: files already in place; an attached zip; an attached
    *extracted* copy (Kaggle unzips uploads: .../supplementary/Etot_ccpvdz.txt); the ACS direct link."""
    dest = root / "data" / "raw" / "qm7b_multilevel"
    if (dest / "supplementary" / "Etot_ccpvdz.txt").exists():
        log("= QM7b multilevel present"); return dest
    dest.mkdir(parents=True, exist_ok=True)
    z = dest / "ct8b00832_si_001.zip"
    att = find_attached("ct8b00832_si_001.zip")
    if att:
        shutil.copy(att, z); log(f"copied attached {att}"); unzip(z, dest); return dest
    marker = find_attached("Etot_ccpvdz.txt")                      # extracted copy anywhere under the inputs
    if marker:
        src = marker.parent
        shutil.copytree(src, dest / "supplementary", dirs_exist_ok=True)
        log(f"copied attached extracted folder {src}"); return dest
    if download(QM7B_SI, z):
        unzip(z, dest); return dest
    raise RuntimeError("QM7b SI not found: attach ct8b00832_si_001.zip (or its extracted 'supplementary' folder) as a Kaggle "
                       "dataset; download it once by hand from https://pubs.acs.org/doi/10.1021/acs.jctc.8b00832 (Supporting Information).")


def precomputed_w4_grid(root: Path) -> Path | None:
    dest = root / "data" / "processed" / "w4-17_grid.jsonl"
    if dest.exists():
        return dest
    att = find_attached("w4-17_grid.jsonl")
    if att:
        dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy(att, dest); log(f"copied attached precomputed grid {att}"); return dest
    return None


def fetch_all(root: Path = ROOT, skip_multixc: bool = False) -> dict:
    """Fetch every dataset; a failure in one does not stop the others.  Returns and writes the availability flags."""
    status = {}
    steps = [("qm9_g4mp2", fetch_qm9_g4mp2), ("accdb", fetch_accdb), ("qm7b", fetch_qm7b)]
    if not skip_multixc:
        steps.append(("multixc", fetch_multixc))
    for name, fn in steps:
        try:
            fn(root); status[name] = True
        except Exception as exc:
            status[name] = False; log(f"!! {name} unavailable: {exc}")
    g = precomputed_w4_grid(root); status["w4_grid_precomputed"] = bool(g)
    log(f"precomputed W4-17 grid: {'yes' if g else 'no (the PySCF campaign will run)'}")
    (root / "data" / "processed").mkdir(parents=True, exist_ok=True)
    (root / "data" / "processed" / "AVAILABLE.json").write_text(json.dumps(status, indent=1))
    log("availability: " + ", ".join(f"{k}={'ok' if v else 'MISSING'}" for k, v in status.items()))
    return status


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=str(ROOT))
    ap.add_argument("--skip-multixc", action="store_true")
    a = ap.parse_args()
    fetch_all(Path(a.root), a.skip_multixc)


In [ ]:
%%writefile {CL_ROOT}/src/conformalladder/palette.py
"""Colour-blind-safe Wong palette and the figure export helper used by every figure script.

Series roles used across the figures (kept as the historical names so the scripts read naturally):
    BLUE   -> Wong blue       #0072B2
    ORANGE -> Wong orange     #E69F00
    AQUA   -> Wong green      #009E73
    YELLOW -> Wong purple     #CC79A7   (4th categorical slot; Wong yellow is too faint for bars)
    SKY    -> Wong sky blue   #56B4E9
    VERM   -> Wong vermillion #D55E00   (warnings / failed FDP labels)
    GRAY   -> neutral grey    #8C8C8C   (baselines: "none", raw)
    INK    -> black           #000000
    MUTED  -> dark grey       #555555
"""
from __future__ import annotations

from pathlib import Path

BLUE, ORANGE, AQUA, YELLOW = "#0072B2", "#E69F00", "#009E73", "#CC79A7"
SKY, VERM, GRAY, INK, MUTED = "#56B4E9", "#D55E00", "#8C8C8C", "#000000", "#555555"
LIGHT_GRAY = "#C8C8C8"
FACE = "white"
DPI = 300


def save_fig(fig, path, dpi: int = DPI):
    """Export PNG and PDF at 300 dpi next to each other (house rule).
    With CL_FIG_NO_SUPTITLE=1 the figure-level title is hidden (journal figures carry it in the caption)."""
    import os
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if os.environ.get("CL_FIG_NO_SUPTITLE") == "1" and getattr(fig, "_suptitle", None) is not None:
        fig._suptitle.set_visible(False)
        fig.subplots_adjust(top=min(0.94, fig.subplotpars.top + 0.06))
    fig.savefig(path.with_suffix(".png"), dpi=dpi, facecolor=FACE)
    fig.savefig(path.with_suffix(".pdf"), dpi=dpi, facecolor=FACE)
    return path.with_suffix(".png")


In [ ]:
%%writefile {CL_ROOT}/notebooks/pilot0_qm9_g4mp2.py
"""Pilot 0 — one rung of the ladder, end to end, on QM9-G4MP2.

Question answered: given a *cheap* B3LYP/6-31G(2df,p) calculation, can we
(1) predict its error against G4MP2 (electronic atomization energy),
(2) attach a distribution-free 90 % interval whose coverage actually holds,
(3) certify, with FDR <= 10 %, the molecules whose corrected value is within
    1 kcal/mol of G4MP2 (these stay cheap; the rest are escalated), and
(4) see what happens to (2)–(3) under covariate shift (size, element) and whether
    weighted conformal repairs it.

Run:  PYTHONPATH=src python notebooks/pilot0_qm9_g4mp2.py
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

from conformalladder import conformal as cp
from conformalladder.data_qm9g4mp2 import FEATURES, build

ALPHA = 0.10          # 90 % intervals
Q_FDR = 0.10          # FDR level for certification
THRESH = 1.0          # kcal/mol "chemical accuracy" target
SEED = 20260920
RESULTS = Path(__file__).resolve().parents[1] / "results" / "pilot0"
RESULTS.mkdir(parents=True, exist_ok=True)


def fit_models(X, y, seed=SEED):
    """Point model mu, dispersion model sigma (on out-of-fold |residual|), CQR quantile models."""
    hp = dict(max_iter=400, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=40,
              l2_regularization=1.0, random_state=seed)
    mu = HistGradientBoostingRegressor(**hp).fit(X, y)
    oof = cross_val_predict(HistGradientBoostingRegressor(**hp), X, y, cv=3)
    sigma = HistGradientBoostingRegressor(**hp).fit(X, np.abs(y - oof))
    qlo = HistGradientBoostingRegressor(loss="quantile", quantile=ALPHA / 2, **hp).fit(X, y)
    qhi = HistGradientBoostingRegressor(loss="quantile", quantile=1 - ALPHA / 2, **hp).fit(X, y)
    return mu, sigma, qlo, qhi


def density_ratio_weights(X_cal, X_test):
    """w(x) ~ p_test(x)/p_cal(x) from a probabilistic classifier (Tibshirani et al. 2019).
    No clipping on purpose: a test point the classifier can separate perfectly from the
    calibration set gets w = inf, which makes the weighted quantile infinite, i.e. the
    method *abstains* instead of pretending to have a guarantee."""
    Xs = np.vstack([X_cal, X_test])
    lab = np.r_[np.zeros(len(X_cal)), np.ones(len(X_test))]
    # cross-fitted probabilities from a flexible classifier (a linear model cannot represent a
    # shift such as "n_heavy == 9"); cross-fitting avoids over-confident in-sample weights
    clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=31,
                                         min_samples_leaf=50, random_state=SEED)
    p = cross_val_predict(clf, Xs, lab, cv=5, method="predict_proba")[:, 1]
    prior = len(X_test) / len(Xs)
    with np.errstate(divide="ignore", invalid="ignore"):
        w = (p / (1 - p)) * ((1 - prior) / prior)
    w = np.where(p >= 1 - 1e-12, np.inf, w)
    w_cal = np.clip(w[: len(X_cal)], 1e-8, 1e8)          # calibration weights must stay finite
    w_test = np.where(w[len(X_cal):] > 1e6, np.inf, w[len(X_cal):])
    return w_cal, w_test


def run_split(name, df, idx_train, idx_cal, idx_test, groups_col="n_heavy_atoms"):
    t0 = time.time()
    X = df[FEATURES].to_numpy(float)
    y = df["eps_kcal"].to_numpy(float)
    Xtr, ytr = X[idx_train], y[idx_train]
    Xca, yca = X[idx_cal], y[idx_cal]
    Xte, yte = X[idx_test], y[idx_test]

    mu, sigma, qlo, qhi = fit_models(Xtr, ytr)
    mu_ca, mu_te = mu.predict(Xca), mu.predict(Xte)
    sg_ca, sg_te = np.maximum(sigma.predict(Xca), 1e-3), np.maximum(sigma.predict(Xte), 1e-3)

    res = {"split": name, "n_train": len(idx_train), "n_cal": len(idx_cal), "n_test": len(idx_test)}
    res["raw_error"] = {"MAE_B3LYP_vs_G4MP2": float(np.mean(np.abs(yte))),
                        "frac_within_1kcal_raw": float(np.mean(np.abs(yte) < THRESH))}
    r_te = yte - mu_te
    res["corrected_error"] = {"MAE": float(np.mean(np.abs(r_te))),
                              "RMSE": float(np.sqrt(np.mean(r_te ** 2))),
                              "frac_within_1kcal": float(np.mean(np.abs(r_te) < THRESH))}

    # --- intervals: scaled residual (split conformal) and CQR --------------------------
    lo, hi, q = cp.scaled_residual_intervals(mu_ca, sg_ca, yca, mu_te, sg_te, ALPHA)
    res["scaled_split_conformal"] = {"q": float(q), "coverage": cp.coverage(yte, lo, hi),
                                     "mean_width": float(np.mean(hi - lo)),
                                     "by_group": cp.group_coverage(yte, lo, hi, df[groups_col].to_numpy()[idx_test])}
    lo2, hi2, q2 = cp.cqr_intervals(qlo.predict(Xca), qhi.predict(Xca), yca,
                                    qlo.predict(Xte), qhi.predict(Xte), ALPHA)
    res["CQR"] = {"q": float(q2), "coverage": cp.coverage(yte, lo2, hi2),
                  "mean_width": float(np.mean(hi2 - lo2)),
                  "by_group": cp.group_coverage(yte, lo2, hi2, df[groups_col].to_numpy()[idx_test])}

    # --- weighted conformal (only meaningful under shift, harmless otherwise) -------------
    w_ca, w_te = density_ratio_weights(Xca, Xte)
    lo3, hi3, q3 = cp.weighted_scaled_intervals(mu_ca, sg_ca, yca, w_ca, mu_te, sg_te, w_te, ALPHA)
    finite = np.isfinite(hi3)
    res["weighted_split_conformal"] = {
        "coverage_all(abstain=miss)": cp.coverage(yte, lo3, hi3),
        "coverage_among_answered": cp.coverage(yte[finite], lo3[finite], hi3[finite]) if finite.any() else None,
        "frac_abstained": float(np.mean(~finite)),
        "mean_width_answered": float(np.mean((hi3 - lo3)[finite])) if finite.any() else None,
        "median_q": float(np.median(q3[finite])) if finite.any() else None,
        "by_group": cp.group_coverage(yte, lo3, hi3, df[groups_col].to_numpy()[idx_test])}

    # --- certification with FDR control (clipped-score conformal p-values + BH) ------------
    pv = cp.conformal_pvalues_small_error(np.abs(yca - mu_ca), sg_ca, sg_te, THRESH,
                                          rng=np.random.default_rng(SEED))
    sel = cp.benjamini_hochberg(pv, Q_FDR)
    res["certification_BH"] = cp.selection_report(sel, np.abs(r_te), THRESH)
    # naive plug-in rule for contrast (no guarantee): certify if the 90 % half-width < 1 kcal/mol
    naive = (q * sg_te) < THRESH
    res["certification_naive_plugin"] = cp.selection_report(naive, np.abs(r_te), THRESH)
    # routing under shift: abstain (escalate) whenever weighted conformal returns an infinite interval
    sel_routed = sel & finite
    res["certification_BH_with_abstention"] = cp.selection_report(sel_routed, np.abs(r_te), THRESH)
    # FDP as a function of q (for the figure)
    res["fdp_curve"] = [{"q": float(qq), **cp.selection_report(cp.benjamini_hochberg(pv, qq), np.abs(r_te), THRESH)}
                        for qq in (0.01, 0.02, 0.05, 0.1, 0.2, 0.3)]
    res["seconds"] = round(time.time() - t0, 1)

    keep = pd.DataFrame({"index": df["index"].to_numpy()[idx_test], "eps": yte, "mu": mu_te,
                         "sigma": sg_te, "lo": lo, "hi": hi, "lo_w": lo3, "hi_w": hi3,
                         "pval": pv, "certified": sel,
                         "n_heavy": df["n_heavy_atoms"].to_numpy()[idx_test]})
    keep.to_parquet(RESULTS / f"test_predictions_{name}.parquet", index=False)
    return res


def main():
    df = build()
    rng = np.random.default_rng(SEED)
    n = len(df)
    all_idx = np.arange(n)
    n_heavy = df["n_heavy_atoms"].to_numpy()
    has_F = (df["n_F"] > 0).to_numpy()
    out = {}

    # A. exchangeable random split 60/20/20
    perm = rng.permutation(all_idx)
    a, b = int(0.6 * n), int(0.8 * n)
    out["A_random"] = run_split("A_random", df, perm[:a], perm[a:b], perm[b:])
    print(json.dumps({k: v for k, v in out["A_random"].items() if "by_group" not in str(k)}, indent=1)[:1500])

    # B1. soft size shift (overlapping support): train/cal drawn from a distribution that keeps
    #     only 10 % of the 9-heavy-atom molecules; test drawn from the natural QM9 distribution.
    keep_prob = np.where(n_heavy == 9, 0.10, 1.0)
    biased = all_idx[rng.uniform(size=n) < keep_prob]
    biased = rng.permutation(biased)
    rest = np.setdiff1d(all_idx, biased)
    cut = int(0.7 * len(biased))
    test_b1 = rng.choice(rest, size=20000, replace=False)
    out["B1_soft_size_shift"] = run_split("B1_soft_size_shift", df, biased[:cut], biased[cut:], test_b1)

    # B2. hard size extrapolation (disjoint support): train/cal on <= 8 heavy atoms, test on 9.
    small = all_idx[n_heavy <= 8]
    big = all_idx[n_heavy == 9]
    perm_s = rng.permutation(small)
    cut = int(0.7 * len(perm_s))
    test_b2 = rng.choice(big, size=min(20000, len(big)), replace=False)
    out["B2_hard_size_extrapolation"] = run_split("B2_hard_size_extrapolation", df, perm_s[:cut], perm_s[cut:], test_b2)

    # C. element shift: train + calibrate on F-free molecules, test on F-containing molecules
    noF = rng.permutation(all_idx[~has_F])
    withF = all_idx[has_F]
    cut = int(0.75 * len(noF))
    out["C_element_shift_F"] = run_split("C_element_shift_F", df, noF[:cut][:60000], noF[cut:][:20000], withF,
                                         groups_col="n_F")

    (RESULTS / "pilot0_results.json").write_text(json.dumps(out, indent=1))
    print("saved", RESULTS / "pilot0_results.json")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CL_ROOT}/notebooks/pilot0_figure.py
"""Summary figure for pilot 0 (reads results/pilot0/*)."""
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "pilot0"
r = json.load(open(RES / "pilot0_results.json"))

from conformalladder.palette import BLUE, ORANGE, AQUA, YELLOW, SKY, VERM, GRAY, INK, MUTED, LIGHT_GRAY, FACE, save_fig
plt.rcParams.update({"font.size": 9, "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "semibold", "axes.titlesize": 10})

fig, ax = plt.subplots(2, 2, figsize=(11.5, 8), facecolor=FACE)
fig.subplots_adjust(hspace=0.55, wspace=0.3, left=0.07, right=0.98, top=0.9, bottom=0.1)

# (a) raw vs corrected error distribution, exchangeable split
a = ax[0, 0]
t = pd.read_parquet(RES / "test_predictions_A_random.parquet")
bins = np.linspace(-20, 20, 121)
a.hist(t["eps"], bins=bins, color=BLUE, alpha=0.85, lw=0, label="raw: B3LYP/6-31G(2df,p) vs G4MP2")
a.hist(t["eps"] - t["mu"], bins=bins, color=ORANGE, alpha=0.85, lw=0, label="after learned correction")
a.axvspan(-1, 1, color=GRAY, alpha=0.15, lw=0)
a.text(1.3, a.get_ylim()[1] * 0.35, "±1 kcal/mol", color=MUTED, fontsize=8)
a.set_xlabel("error in electronic atomization energy (kcal/mol)")
a.set_ylabel("molecules")
a.set_title("(a) Error of a cheap calculation before / after learned correction")
a.legend(frameon=False, fontsize=8, loc="upper left")
a.grid(axis="y", color="#eeede9", lw=0.6); a.set_axisbelow(True)
ra, ca = r["A_random"]["raw_error"]["MAE_B3LYP_vs_G4MP2"], r["A_random"]["corrected_error"]["MAE"]
a.text(6, a.get_ylim()[1] * 0.45, f"MAE {ra:.2f} → {ca:.2f} kcal/mol\n(26 052 test molecules)", color=INK, fontsize=8.5)

# (b) coverage of nominal-90 % intervals per scenario
b = ax[0, 1]
scen = ["A_random", "B1_soft_size_shift", "B2_hard_size_extrapolation", "C_element_shift_F"]
labels = ["exchangeable\nrandom split", "soft size shift\noverlapping support", "hard extrapolation\n≤8 → 9 heavy atoms", "new element\nno F → F"]
x = np.arange(len(scen)); w = 0.26
cov_split = [r[s]["scaled_split_conformal"]["coverage"] for s in scen]
cov_cqr = [r[s]["CQR"]["coverage"] for s in scen]
cov_w, abst = [], []
for s in scen:
    d = r[s]["weighted_split_conformal"]
    cov_w.append(d["coverage_among_answered"] if d["coverage_among_answered"] is not None else 0.0)
    abst.append(d["frac_abstained"])
b.bar(x - w, cov_split, w, color=BLUE, lw=0, label="split conformal (scaled residual)")
b.bar(x, cov_cqr, w, color=ORANGE, lw=0, label="CQR")
bars = b.bar(x + w, cov_w, w, color=AQUA, lw=0, label="weighted conformal (answered only)")
for i, (bar, fa) in enumerate(zip(bars, abst)):
    if fa >= 0.999:
        b.text(bar.get_x() + bar.get_width() / 2, 0.03, "abstains\n100 %", ha="center", va="bottom", fontsize=7.5, color=AQUA, fontweight="semibold")
for i, v in enumerate(cov_split):
    b.text(x[i] - w, v + 0.015, f"{v:.2f}", ha="center", fontsize=7.5, color=MUTED)
b.axhline(0.9, color=INK, lw=1, ls="--"); b.text(3.45, 0.92, "target 0.90", ha="right", fontsize=8, color=INK)
b.set_xticks(x); b.set_xticklabels(labels, fontsize=7.2)
b.set_ylim(0, 1.42); b.set_ylabel("empirical coverage of 90 % intervals")
b.set_title("(b) Coverage of 90 % intervals under distribution shift")
b.legend(frameon=False, fontsize=7.5, loc="upper center", ncol=1, bbox_to_anchor=(0.5, 1.0))
b.grid(axis="y", color="#eeede9", lw=0.6); b.set_axisbelow(True)

# (c) FDR control: empirical FDP vs nominal q (exchangeable split)
c = ax[1, 0]
curve = r["A_random"]["fdp_curve"]
qs = [d["q"] for d in curve]; fdp = [d["FDP"] for d in curve]; frac = [d["fraction_selected"] for d in curve]
c.plot([0, 0.32], [0, 0.32], color=GRAY, lw=1, ls="--"); c.text(0.25, 0.27, "FDP = q", color=MUTED, fontsize=8, rotation=38)
c.plot(qs, fdp, color=BLUE, lw=2, marker="o", ms=5, label="conformal p-values + BH (empirical FDP)")
for qq, f, fr in zip(qs, fdp, frac):
    if qq < 0.05: continue
    c.annotate(f"{100*fr:.0f}% certified", (qq, f), textcoords="offset points", xytext=(6, -11), fontsize=7, color=MUTED)
c.set_xlabel("nominal FDR level q"); c.set_ylabel("empirical false-discovery proportion")
c.set_xlim(0, 0.32); c.set_ylim(0, 0.32)
c.set_title("(c) FDR-controlled certification of |error| < 1 kcal/mol")
c.legend(frameon=False, fontsize=8, loc="upper left")
c.grid(color="#eeede9", lw=0.6); c.set_axisbelow(True)

# (d) certified fraction and FDP: BH vs naive rule, per scenario
d = ax[1, 1]
bh = [r[s]["certification_BH_with_abstention"] for s in scen]
nv = [r[s]["certification_naive_plugin"] for s in scen]
d.bar(x - w / 2, [v["fraction_selected"] for v in bh], w, color=BLUE, lw=0, label="BH at q = 0.10 (+ abstain if OOD)")
d.bar(x + w / 2, [v["fraction_selected"] for v in nv], w, color=ORANGE, lw=0, label="naive rule: half-width < 1 kcal/mol")
for i in range(len(scen)):
    d.text(x[i] - w / 2, bh[i]["fraction_selected"] + 0.006, f"FDP {bh[i]['FDP']:.2f}", ha="center", fontsize=7, color=INK)
    d.text(x[i] + w / 2, nv[i]["fraction_selected"] + 0.006, f"FDP {nv[i]['FDP']:.2f}", ha="center", fontsize=7, color=INK)
d.set_xticks(x); d.set_xticklabels(labels, fontsize=7.2)
d.set_ylabel("fraction of molecules certified (kept cheap)")
d.set_ylim(0, 0.22)
d.set_title("(d) Routing: certified molecules stay cheap, the rest escalate")
d.legend(frameon=False, fontsize=7.5, loc="upper right")
d.grid(axis="y", color="#eeede9", lw=0.6); d.set_axisbelow(True)

fig.suptitle("ConformalLadder — pilot 0 on QM9-G4MP2 (130 258 molecules; features from the cheap calculation only)",
             fontsize=11, fontweight="semibold", color=INK)
save_fig(fig, RES / "pilot0_summary.png")
print("saved", RES / "pilot0_summary.png")


In [ ]:
%%writefile {CL_ROOT}/notebooks/pilot1_multixc_isomers.py
"""Pilot 1 — the ladder on MultiXC-QM9 isomerization energies (76 functionals x SZ/DZP/TZP).

Why isomerization energies: MultiXC-QM9 atomization energies carry large per-element constant
offsets that depend on the functional (an atomic-reference artefact of the post-SCF ADF protocol;
see results/multixc_functional_survey_atomref.csv).  For constitutional isomers the atomic
references cancel exactly, so dE(A -> B) is clean at every level, and G4MP2 gives a ~1 kcal/mol
reference.  Splits are *formula-grouped* (all isomers of a formula fall in one fold), so no molecule
is shared between training, calibration and test reactions.

Ladder: cheap = PBE/DZP  ->  PBE/TZP (basis)  ->  best/TZP (functional)  ->  best-D/TZP (dispersion)
        ->  G4MP2 (remainder).  'best' is chosen on the training fold among functionals that have a
        '-D' twin in the dataset, so every layer is populated.

Run: OMP_NUM_THREADS=1 PYTHONPATH=src python notebooks/pilot1_multixc_isomers.py
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

from conformalladder import conformal as cp
from conformalladder import layers as L
from conformalladder.data_multixc import build, functionals
from conformalladder.data_qm9g4mp2 import BOND_KEYS

ALPHA, Q_FDR, THRESH, SEED = 0.10, 0.10, 1.0, 20260920
ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "pilot1_multixc"; RES.mkdir(parents=True, exist_ok=True)
XC_C, B_C, B_BIG = "PBE", "DZP", "TZP"
D_TWINS = {"BLYP": "BLYP-D", "BP": "BP86-D", "PBE": "PBE-D", "TPSS": "TPSS-D", "B3LYP(VWN5)": "B3LYP-D"}
CHEAP_GGAS = ["PBE", "BLYP", "BP", "PW91", "RPBE", "OLYP", "TPSS", "M06-L", "REVPBE", "HCTH/407"]
PROPS = ["homo", "lumo", "bandgap", "mu", "alpha", "R2", "zpe", "cv"]
LAYERS = ["eps_total", "beta_basis", "phi_functional", "delta_dispersion", "rho_remainder", "I_bf"]
MAX_PAIRS_PER_FORMULA = 60


def build_pairs(m: pd.DataFrame, rng) -> pd.DataFrame:
    fx = functionals(m)
    ecols = [f"{f}_{b}" for f in fx for b in ("SZ", "DZP", "TZP")] + ["GFNXTB"]
    rows_a, rows_b = [], []
    for formula, g in m.groupby("chemformula"):
        k = len(g)
        if k < 2:
            continue
        ii = g.index.to_numpy()
        n_pairs = min(MAX_PAIRS_PER_FORMULA, k * (k - 1) // 2)
        seen = set()
        while len(seen) < n_pairs:
            a, b = rng.choice(k, 2, replace=False)
            key = (min(a, b), max(a, b))
            seen.add(key)
        for a, b in seen:
            rows_a.append(ii[a]); rows_b.append(ii[b])
    A = m.loc[rows_a].reset_index(drop=True); B = m.loc[rows_b].reset_index(drop=True)
    out = pd.DataFrame({"rid": [f"iso_{i}" for i in range(len(A))], "formula": A["chemformula"].values,
                        "index_A": A["index"].values, "index_B": B["index"].values,
                        "n_heavy": A["n_heavy_atoms"].values, "has_F": (A["n_F"].values > 0),
                        "ref": (B["De_g4mp2_kcal"] - A["De_g4mp2_kcal"]).values})
    for c in ecols:
        f, _, b = c.rpartition("_") if c != "GFNXTB" else ("GFNXTB", "", "none")
        out[f"{f}|{b}"] = (B[c] - A[c]).values
    for f, d in D_TWINS.items():
        out[f"disp:{f}+D"] = out[f"{d}|TZP"] - out[f"{f}|TZP"]
    # cheap features: QM9 property differences, bond-count differences, composition, cheap-level energies
    for p in PROPS:
        out[f"d_{p}"] = (B[p] - A[p]).values
    for k in BOND_KEYS + ["n_rings", "n_arom_rings", "n_rot_bonds"]:
        out[f"d_{k}"] = (B[k] - A[k]).values
        out[f"s_{k}"] = (B[k] + A[k]).values
    for el in ["C", "H", "N", "O", "F"]:
        out[f"n_{el}"] = A[f"n_{el}"].values
    out["subset"] = "n_heavy=" + out["n_heavy"].astype(str)
    return out


def fit_mu_sigma(X, y):
    hp = dict(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=30, l2_regularization=1.0, random_state=SEED)
    mu = HistGradientBoostingRegressor(**hp).fit(X, y)
    oof = cross_val_predict(HistGradientBoostingRegressor(**hp), X, y, cv=4)
    sigma = HistGradientBoostingRegressor(**hp).fit(X, np.abs(y - oof))
    return mu, sigma


def density_ratio_weights(X_cal, X_test):
    Xs = np.vstack([X_cal, X_test]); lab = np.r_[np.zeros(len(X_cal)), np.ones(len(X_test))]
    clf = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=50, random_state=SEED)
    p = cross_val_predict(clf, Xs, lab, cv=5, method="predict_proba")[:, 1]
    prior = len(X_test) / len(Xs)
    with np.errstate(divide="ignore", invalid="ignore"):
        w = (p / (1 - p)) * ((1 - prior) / prior)
    w = np.where(p >= 1 - 1e-12, np.inf, w)
    return np.clip(w[: len(X_cal)], 1e-8, 1e8), np.where(w[len(X_cal):] > 1e6, np.inf, w[len(X_cal):])


def evaluate(name, re, feats, X, i_tr, i_ca, i_te, xc_best):
    budget = L.layer_budget(re, XC_C, B_C, xc_best, B_BIG, disp_tag="D")
    assert np.nanmax(np.abs(budget["check_identity"])) < 1e-8
    out = {"split": name, "n_train": len(i_tr), "n_cal": len(i_ca), "n_test": len(i_te), "layers": {}}
    w_ca = w_te = None
    for layer in LAYERS:
        y = budget[layer].to_numpy(float)
        mu, sg = fit_mu_sigma(X[i_tr], y[i_tr])
        m_ca, s_ca = mu.predict(X[i_ca]), np.maximum(sg.predict(X[i_ca]), 1e-3)
        m_te, s_te = mu.predict(X[i_te]), np.maximum(sg.predict(X[i_te]), 1e-3)
        lo, hi, q = cp.scaled_residual_intervals(m_ca, s_ca, y[i_ca], m_te, s_te, ALPHA)
        rec = {"MAE_raw": float(np.mean(np.abs(y[i_te]))), "MAE_after_model": float(np.mean(np.abs(y[i_te] - m_te))),
               "coverage": cp.coverage(y[i_te], lo, hi), "mean_width": float(np.mean(hi - lo)), "q": float(q),
               "by_group": cp.group_coverage(y[i_te], lo, hi, re["subset"].to_numpy()[i_te])}
        if layer == "eps_total":
            if w_ca is None:
                w_ca, w_te = density_ratio_weights(X[i_ca], X[i_te])
            lo3, hi3, _ = cp.weighted_scaled_intervals(m_ca, s_ca, y[i_ca], w_ca, m_te, s_te, w_te, ALPHA)
            finite = np.isfinite(hi3)
            rec["weighted_conformal"] = {"frac_abstained": float(np.mean(~finite)),
                                         "coverage_among_answered": cp.coverage(y[i_te][finite], lo3[finite], hi3[finite]) if finite.any() else None}
            pv = cp.conformal_pvalues_small_error(np.abs(y[i_ca] - m_ca), s_ca, s_te, THRESH, rng=np.random.default_rng(SEED))
            sel = cp.benjamini_hochberg(pv, Q_FDR)
            rec["certification_BH"] = cp.selection_report(sel, np.abs(y[i_te] - m_te), THRESH)
            rec["certification_BH_with_abstention"] = cp.selection_report(sel & finite, np.abs(y[i_te] - m_te), THRESH)
            rec["certification_naive_plugin"] = cp.selection_report((q * s_te) < THRESH, np.abs(y[i_te] - m_te), THRESH)
            rec["frac_raw_within_1kcal"] = float(np.mean(np.abs(y[i_te]) < THRESH))
            rec["fdp_curve"] = [{"q": float(qq), **cp.selection_report(cp.benjamini_hochberg(pv, qq) & finite, np.abs(y[i_te] - m_te), THRESH)}
                                for qq in (0.02, 0.05, 0.1, 0.2, 0.3)]
        out["layers"][layer] = rec
    return out, budget


def main():
    t0 = time.time()
    rng = np.random.default_rng(SEED)
    m = build()
    pairs_path = RES / "isomer_pairs.parquet"
    if pairs_path.exists():
        re = pd.read_parquet(pairs_path)
    else:
        re = build_pairs(m, rng); re.to_parquet(pairs_path, index=False)
    print(f"isomer pairs: {len(re)} from {re['formula'].nunique()} formulas  ({time.time()-t0:.0f}s)")

    feat_cols = ([f"{XC_C}|{B_C}", "GFNXTB|none"] + [f"{f}|{B_C}" for f in CHEAP_GGAS if f != XC_C]
                 + [f"d_{p}" for p in PROPS] + [c for c in re.columns if c.startswith("d_") and c[2:] in BOND_KEYS + ["n_rings", "n_arom_rings", "n_rot_bonds"]]
                 + [c for c in re.columns if c.startswith("s_")] + [f"n_{el}" for el in ["C", "H", "N", "O", "F"]] + ["n_heavy"])
    X = re[feat_cols].to_numpy(float)
    X[:, 2:2 + len(CHEAP_GGAS) - 1] -= X[:, [0]]          # other GGAs as deviations from the cheap level
    spread = re[[f"{f}|{B_C}" for f in CHEAP_GGAS]].to_numpy(float)
    X = np.column_stack([X, spread.max(1) - spread.min(1), spread.std(1)])
    feat_cols += ["gga_spread_range", "gga_spread_std"]

    # formula-grouped splits
    formulas = re["formula"].to_numpy(); uf = np.unique(formulas); rng.shuffle(uf)
    n_heavy = re["n_heavy"].to_numpy(); has_F = re["has_F"].to_numpy()
    fold = {f: i for i, f in enumerate(uf)}
    fid = np.array([fold[f] for f in formulas]); nf = len(uf)
    i_tr = np.where(fid < 0.5 * nf)[0]; i_ca = np.where((fid >= 0.5 * nf) & (fid < 0.75 * nf))[0]; i_te = np.where(fid >= 0.75 * nf)[0]

    # choose the best D-capable functional on the training fold (MAE of best-D/TZP vs G4MP2)
    cand = {f: float(np.mean(np.abs(re["ref"].values[i_tr] - re[f"{D_TWINS[f]}|TZP"].values[i_tr]))) for f in D_TWINS}
    xc_best = min(cand, key=cand.get)
    overall = sorted(((float(np.mean(np.abs(re["ref"].values[i_tr] - re[f"{f}|TZP"].values[i_tr]))), f) for f in functionals(m)))[:8]
    print("D-capable candidates (train MAE, kcal/mol):", {k: round(v, 2) for k, v in cand.items()}, "-> best:", xc_best)
    print("best 8 functionals overall at TZP (train MAE):", [(f, round(v, 2)) for v, f in overall])

    results = {"cheap": [XC_C, B_C], "big_basis": B_BIG, "best_functional": xc_best, "best_functional_D": D_TWINS[xc_best],
               "train_MAE_candidates": cand, "best_overall_TZP": overall, "n_pairs": int(len(re)), "features": feat_cols, "splits": {}}
    res_a, budget = evaluate("A_formula_grouped_random", re, None, X, i_tr, i_ca, i_te, xc_best)
    results["splits"]["A_formula_grouped_random"] = res_a
    budget.to_csv(RES / "budget_A.csv", index=False)
    L.summarize_budget(budget.assign(subset=re["subset"])).to_csv(RES / "budget_summary_A.csv", index=False)

    # B: size extrapolation (train/cal on n_heavy <= 8, test on n_heavy == 9)
    small = np.where(n_heavy <= 8)[0]; big = np.where(n_heavy == 9)[0]
    fs = np.unique(formulas[small]); rng.shuffle(fs); fs_id = {f: i for i, f in enumerate(fs)}
    sid = np.array([fs_id.get(f, -1) for f in formulas])
    tr_b = np.where((sid >= 0) & (sid < 0.65 * len(fs)))[0]; ca_b = np.where(sid >= 0.65 * len(fs))[0]
    te_b = rng.choice(big, size=min(20000, len(big)), replace=False)
    res_b, _ = evaluate("B_size_extrapolation_le8_to_9", re, None, X, tr_b, ca_b, te_b, xc_best)
    results["splits"]["B_size_extrapolation_le8_to_9"] = res_b

    # C: element shift (no F -> F)
    noF = np.where(~has_F)[0]; withF = np.where(has_F)[0]
    fs = np.unique(formulas[noF]); rng.shuffle(fs); fs_id = {f: i for i, f in enumerate(fs)}
    sid = np.array([fs_id.get(f, -1) for f in formulas])
    tr_c = np.where((sid >= 0) & (sid < 0.7 * len(fs)))[0]; ca_c = np.where(sid >= 0.7 * len(fs))[0]
    res_c, _ = evaluate("C_element_shift_noF_to_F", re, None, X, tr_c, ca_c, withF, xc_best)
    results["splits"]["C_element_shift_noF_to_F"] = res_c

    (RES / "pilot1_results.json").write_text(json.dumps(results, indent=1))
    for name, r in results["splits"].items():
        print("=" * 90); print(name, "| n", r["n_train"], r["n_cal"], r["n_test"])
        for k, v in r["layers"].items():
            print(f"  {k:18s} MAE {v['MAE_raw']:6.2f} -> {v['MAE_after_model']:5.2f} | coverage {v['coverage']:.3f} width {v['mean_width']:.2f}")
        e = r["layers"]["eps_total"]
        print("  raw within 1 kcal/mol: %.1f%% | weighted conformal: abstain %.1f%%, coverage answered %s" % (100 * e["frac_raw_within_1kcal"], 100 * e["weighted_conformal"]["frac_abstained"], e["weighted_conformal"]["coverage_among_answered"]))
        for kk in ("certification_BH", "certification_BH_with_abstention", "certification_naive_plugin"):
            s = e[kk]; print(f"  {kk:34s}: certified {100*s['fraction_selected']:5.1f}% FDP {s['FDP']:.3f} power {s['power']:.3f} base {s['base_rate_good']:.3f}")
    print(f"total {time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CL_ROOT}/notebooks/pilot1_figure.py
"""Figure for pilot 1 (MultiXC-QM9 isomerization ladder)."""
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "pilot1_multixc"
r = json.load(open(RES / "pilot1_results.json"))
s = pd.read_csv(RES / "budget_summary_A.csv")
b = pd.read_csv(RES / "budget_A.csv")

from conformalladder.palette import BLUE, ORANGE, AQUA, YELLOW, SKY, VERM, GRAY, INK, MUTED, LIGHT_GRAY, FACE, save_fig
plt.rcParams.update({"font.size": 9, "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "semibold", "axes.titlesize": 10})
fig, ax = plt.subplots(2, 2, figsize=(11.5, 8.2), facecolor=FACE)
fig.subplots_adjust(hspace=0.5, wspace=0.3, left=0.07, right=0.98, top=0.87, bottom=0.1)
xc_c, b_c = r["cheap"]; xc_b, b_big = r["best_functional"], r["big_basis"]
lay = ["eps_total", "beta_basis", "phi_functional", "delta_dispersion", "rho_remainder", "I_bf"]
names = {"eps_total": "total ε", "beta_basis": "basis β", "phi_functional": "functional φ", "delta_dispersion": "dispersion δ", "rho_remainder": "remainder ρ", "I_bf": "interaction"}

# (a) layer MAE vs molecule size
a = ax[0, 0]
ss = s[s["subset"] != "ALL"].copy(); ss["nh"] = ss["subset"].str.replace("n_heavy=", "").astype(int); ss = ss[ss["n"] >= 100].sort_values("nh")
x = np.arange(len(ss)); w = 0.18
for k, (col, lab, colr) in enumerate([("beta_basis_MAE", f"basis {b_c}→{b_big}", BLUE), ("phi_functional_MAE", f"functional {xc_c}→B3LYP", ORANGE),
                                       ("delta_dispersion_MAE", "dispersion (-D)", AQUA), ("rho_remainder_MAE", "remainder to G4MP2", YELLOW)]):
    a.bar(x + (k - 1.5) * w, ss[col], w, color=colr, lw=0, label=lab)
a.plot(x, ss["eps_total_MAE"], "o", color=INK, ms=5, label=f"total error of {xc_c}/{b_c}")
a.plot(x, ss["I_bf_MAE"], "s", color=GRAY, ms=5, label="non-additivity |I_bf|")
a.set_xticks(x); a.set_xticklabels([f"{n} heavy\n(n={c})" for n, c in zip(ss["nh"], ss["n"])], fontsize=7.5)
a.set_ylabel("MAE (kcal/mol)"); a.set_title("(a) Layer budget of isomerization energies vs molecule size")
a.legend(frameon=False, fontsize=7.5, ncol=2, loc="upper left"); a.set_ylim(0, 4.6)
a.grid(axis="y", color="#eeede9", lw=0.6); a.set_axisbelow(True)

# (b) I_bf vs phi
bb = ax[0, 1]
bb.scatter(b["phi_functional"], b["I_bf"], s=6, color=BLUE, alpha=0.35, lw=0)
bb.axhline(0, color=GRAY, lw=0.8)
lim = np.nanpercentile(np.abs(b["phi_functional"]), 99.5); bb.set_xlim(-lim, lim)
bb.set_ylim(-1.2, 1.2)
bb.set_xlabel(f"functional correction φ (kcal/mol)"); bb.set_ylabel("basis × functional interaction I_bf (kcal/mol)")
bb.set_title("(b) Non-additivity is small but not zero")
bb.text(0.02, 0.95, f"median |I_bf| = {np.nanmedian(np.abs(b['I_bf'])):.2f} kcal/mol\nmedian |I_bf| / |φ| = {np.nanmedian(np.abs(b['I_bf'])/np.maximum(np.abs(b['phi_functional']),1e-6)):.2f}\n"
        f"|I_bf| > 0.5 kcal/mol in {100*np.nanmean(np.abs(b['I_bf'])>0.5):.1f} % of reactions", transform=bb.transAxes, va="top", fontsize=8)
bb.grid(color="#eeede9", lw=0.6); bb.set_axisbelow(True)

# (c) coverage per layer for the three splits
c = ax[1, 0]
splits = [("A_formula_grouped_random", "formula-grouped random", BLUE), ("B_size_extrapolation_le8_to_9", "size extrapolation ≤8 → 9", ORANGE), ("C_element_shift_noF_to_F", "element shift no F → F", AQUA)]
x = np.arange(len(lay)); w = 0.27
for k, (key, lab, colr) in enumerate(splits):
    cov = [r["splits"][key]["layers"][l]["coverage"] for l in lay]
    c.bar(x + (k - 1) * w, cov, w, color=colr, lw=0, label=lab)
    for i, v in enumerate(cov):
        c.text(x[i] + (k - 1) * w, v + 0.01, f"{v:.2f}", ha="center", fontsize=6.3, color=MUTED, rotation=90)
c.axhline(0.9, color=INK, lw=1, ls="--", label="target 0.90")
c.set_xticks(x); c.set_xticklabels([names[l] for l in lay], fontsize=7.5); c.set_ylim(0, 1.25)
c.set_ylabel("empirical coverage (90 % intervals)"); c.set_title("(c) Coverage per layer and split (plain split conformal)")
c.legend(frameon=False, fontsize=7.5, loc="upper left", ncol=2); c.grid(axis="y", color="#eeede9", lw=0.6); c.set_axisbelow(True)

# (d) certification and FDP
d = ax[1, 1]
x = np.arange(len(splits)); w = 0.26
bh = [r["splits"][k]["layers"]["eps_total"]["certification_BH"] for k, _, _ in splits]
bha = [r["splits"][k]["layers"]["eps_total"]["certification_BH_with_abstention"] for k, _, _ in splits]
nv = [r["splits"][k]["layers"]["eps_total"]["certification_naive_plugin"] for k, _, _ in splits]
for k, (vals, lab, colr) in enumerate([(bh, "BH, q = 0.10", BLUE), (bha, "BH + abstain if OOD", AQUA), (nv, "naive: half-width < 1 kcal/mol", ORANGE)]):
    d.bar(x + (k - 1) * w, [v["fraction_selected"] for v in vals], w, color=colr, lw=0, label=lab)
    for i, v in enumerate(vals):
        d.text(x[i] + (k - 1) * w, v["fraction_selected"] + 0.005, f"FDP\n{v['FDP']:.2f}", ha="center", fontsize=6.5, color=INK)
d.set_xticks(x); d.set_xticklabels([lab for _, lab, _ in splits], fontsize=7.5); d.set_ylim(0, 0.42)
d.set_ylabel("fraction certified within 1 kcal/mol of G4MP2"); d.set_title("(d) FDR-controlled certification of the corrected cheap level")
d.legend(frameon=False, fontsize=7.5, loc="upper right"); d.grid(axis="y", color="#eeede9", lw=0.6); d.set_axisbelow(True)

e = r["splits"]["A_formula_grouped_random"]["layers"]["eps_total"]
fig.suptitle(f"ConformalLadder on MultiXC-QM9 isomerization energies ({r['n_pairs']:,} isomer pairs, G4MP2 reference, formula-grouped splits)\n"
             f"cheap {xc_c}/{b_c} → {xc_c}/{b_big} → B3LYP/{b_big} → B3LYP-D/{b_big} → G4MP2;  total MAE {e['MAE_raw']:.2f} → {e['MAE_after_model']:.2f} kcal/mol",
             fontsize=10.5, fontweight="semibold", color=INK)
save_fig(fig, RES / "pilot1_summary.png")
print("saved", RES / "pilot1_summary.png")


In [ ]:
%%writefile {CL_ROOT}/notebooks/pilot2_qm7b_additivity.py
"""Pilot 2 — correlation x basis non-additivity on the QM7b multilevel grid (7,211 molecules).

The classic composite assumption (G4MP2 / Wn / FPD style) is
    E[CC/big]  ~=  E[CC/small] + ( E[MP2/big] - E[MP2/small] ),
i.e. the coupled-cluster correction beyond MP2 is transferable between bases.  With the full
{HF, MP2, CCSD(T)} x {STO-3G, 6-31G, cc-pVDZ} grid this assumption can be *measured* per molecule:

    I_bc = [CCSD(T) - MP2]_{cc-pVDZ} - [CCSD(T) - MP2]_{6-31G}        (composite additivity error)

Ladder: cheap = MP2/6-31G  ->  MP2/cc-pVDZ (basis beta)  ->  CCSD(T)/cc-pVDZ (correlation kappa);
eps = beta + kappa exactly, and the composite estimate errs by exactly I_bc.
Targets: atomization energies (AE) and isomerization energies (formula-grouped pairs).
We (1) predict every layer from cheap quantities with conformal intervals, and (2) certify with
FDR control the molecules/reactions for which the additive composite is within 1 kcal/mol.

Run: OMP_NUM_THREADS=1 PYTHONPATH=src python notebooks/pilot2_qm7b_additivity.py
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

from conformalladder import conformal as cp
from conformalladder.data_qm7b import build

ALPHA, Q_FDR, THRESH, SEED = 0.10, 0.10, 1.0, 20260920
ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "pilot2_qm7b"; RES.mkdir(parents=True, exist_ok=True)
# cheap information only: HF in every basis, MP2 in the two small bases, CCSD(T) in the minimal basis.
# MP2/cc-pVDZ and CCSD(T)/6-31G are deliberately excluded so that every layer (beta, kappa, I_bc) is a genuine
# prediction target and not a difference of two input features.
CHEAP_LEVELS = ["HF|STO-3G", "HF|6-31G", "HF|cc-pVDZ", "MP2|STO-3G", "MP2|6-31G", "CCSD(T)|STO-3G"]
TARGETS = ["eps_total", "beta_basis", "kappa_correlation", "I_bc"]


def targets_from(E):
    """E: DataFrame with columns 'X|method|basis' (X = AE or dE). Returns target table."""
    t = pd.DataFrame(index=E.index)
    ref = E["CCSD(T)|cc-pVDZ"]
    t["eps_total"] = ref - E["MP2|6-31G"]
    t["beta_basis"] = E["MP2|cc-pVDZ"] - E["MP2|6-31G"]
    t["kappa_correlation"] = ref - E["MP2|cc-pVDZ"]
    t["I_bc"] = (ref - E["MP2|cc-pVDZ"]) - (E["CCSD(T)|6-31G"] - E["MP2|6-31G"])
    assert np.allclose(t["eps_total"], t["beta_basis"] + t["kappa_correlation"])
    return t


def fit_mu_sigma(X, y):
    hp = dict(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20, l2_regularization=1.0, random_state=SEED)
    mu = HistGradientBoostingRegressor(**hp).fit(X, y)
    oof = cross_val_predict(HistGradientBoostingRegressor(**hp), X, y, cv=4)
    sigma = HistGradientBoostingRegressor(**hp).fit(X, np.abs(y - oof))
    return mu, sigma


def density_ratio_weights(X_cal, X_test):
    Xs = np.vstack([X_cal, X_test]); lab = np.r_[np.zeros(len(X_cal)), np.ones(len(X_test))]
    clf = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_leaf_nodes=15, min_samples_leaf=40, random_state=SEED)
    p = cross_val_predict(clf, Xs, lab, cv=5, method="predict_proba")[:, 1]
    prior = len(X_test) / len(Xs)
    with np.errstate(divide="ignore", invalid="ignore"):
        w = (p / (1 - p)) * ((1 - prior) / prior)
    w = np.where(p >= 1 - 1e-12, np.inf, w)
    return np.clip(w[: len(X_cal)], 1e-8, 1e8), np.where(w[len(X_cal):] > 1e6, np.inf, w[len(X_cal):])


def evaluate(name, T, X, groups, i_tr, i_ca, i_te):
    out = {"split": name, "n_train": len(i_tr), "n_cal": len(i_ca), "n_test": len(i_te), "targets": {}}
    w_ca = w_te = None
    for tg in TARGETS:
        y = T[tg].to_numpy(float)
        mu, sg = fit_mu_sigma(X[i_tr], y[i_tr])
        m_ca, s_ca = mu.predict(X[i_ca]), np.maximum(sg.predict(X[i_ca]), 1e-3)
        m_te, s_te = mu.predict(X[i_te]), np.maximum(sg.predict(X[i_te]), 1e-3)
        lo, hi, q = cp.scaled_residual_intervals(m_ca, s_ca, y[i_ca], m_te, s_te, ALPHA)
        rec = {"MAE_raw": float(np.mean(np.abs(y[i_te]))), "MAE_after_model": float(np.mean(np.abs(y[i_te] - m_te))),
               "coverage": cp.coverage(y[i_te], lo, hi), "mean_width": float(np.mean(hi - lo)), "q": float(q),
               "by_group": cp.group_coverage(y[i_te], lo, hi, groups[i_te])}
        if tg == "I_bc":
            # certify that the *additive composite* is within THRESH (raw |I_bc| < 1) and that the
            # *learned* composite (composite + predicted I_bc) is within THRESH
            if w_ca is None:
                w_ca, w_te = density_ratio_weights(X[i_ca], X[i_te])
            lo3, hi3, _ = cp.weighted_scaled_intervals(m_ca, s_ca, y[i_ca], w_ca, m_te, s_te, w_te, ALPHA)
            finite = np.isfinite(hi3)
            rec["weighted_conformal"] = {"frac_abstained": float(np.mean(~finite)),
                                         "coverage_among_answered": cp.coverage(y[i_te][finite], lo3[finite], hi3[finite]) if finite.any() else None}
            # (a) raw additive composite: confidence statistic = |predicted I_bc| + sigma (small = confident that I_bc ~ 0)
            conf_ca, conf_te = np.abs(m_ca) + s_ca, np.abs(m_te) + s_te
            pv = cp.conformal_pvalues_small_error(np.abs(y[i_ca]), conf_ca, conf_te, THRESH, rng=np.random.default_rng(SEED))
            sel = cp.benjamini_hochberg(pv, Q_FDR)
            rec["certify_raw_composite"] = cp.selection_report(sel & finite, np.abs(y[i_te]), THRESH)
            rec["frac_raw_composite_within_1kcal"] = float(np.mean(np.abs(y[i_te]) < THRESH))
            # (b) learned composite
            pv2 = cp.conformal_pvalues_small_error(np.abs(y[i_ca] - m_ca), s_ca, s_te, THRESH, rng=np.random.default_rng(SEED))
            sel2 = cp.benjamini_hochberg(pv2, Q_FDR)
            rec["certify_learned_composite"] = cp.selection_report(sel2 & finite, np.abs(y[i_te] - m_te), THRESH)
            rec["certify_learned_composite_no_gate"] = cp.selection_report(sel2, np.abs(y[i_te] - m_te), THRESH)
            rec["frac_learned_composite_within_1kcal"] = float(np.mean(np.abs(y[i_te] - m_te) < THRESH))
            rec["naive_rule"] = cp.selection_report((q * s_te) < THRESH, np.abs(y[i_te] - m_te), THRESH)
        out["targets"][tg] = rec
    return out


def main():
    t0 = time.time(); rng = np.random.default_rng(SEED)
    d = build()
    bc_cols = [c for c in d.columns if c.startswith("bc_")]
    results = {"n_molecules": int(len(d)), "problems": {}}

    # ---------------- problem 1: atomization energies ----------------
    E = pd.DataFrame({f"{m}|{b}": d[f"AE|{m}|{b}"] for m in ["HF", "MP2", "CCSD(T)"] for b in ["STO-3G", "6-31G", "cc-pVDZ"]})
    T = targets_from(E)
    X = np.column_stack([E[CHEAP_LEVELS].to_numpy(float), d[["n_C", "n_H", "n_N", "n_O", "n_S", "n_Cl", "n_heavy", "n_atom", "Enn"]].to_numpy(float), d[bc_cols].to_numpy(float)])
    groups = ("n_heavy=" + d["n_heavy"].astype(str)).to_numpy()
    idx = rng.permutation(len(d)); a, b = int(0.5 * len(d)), int(0.75 * len(d))
    pr = {"summary_kcal": {tg: {"MAE": float(T[tg].abs().mean()), "mean": float(T[tg].mean()), "P95_abs": float(T[tg].abs().quantile(0.95))} for tg in TARGETS}, "splits": {}}
    pr["splits"]["A_random"] = evaluate("A_random", T, X, groups, idx[:a], idx[a:b], idx[b:])
    hetero = ((d["n_S"] > 0) | (d["n_Cl"] > 0)).to_numpy()
    org = rng.permutation(np.where(~hetero)[0]); c = int(0.7 * len(org))
    pr["splits"]["B_element_shift_to_S_Cl"] = evaluate("B_element_shift_to_S_Cl", T, X, groups, org[:c], org[c:], np.where(hetero)[0])
    small = rng.permutation(np.where(d["n_heavy"] <= 6)[0]); c = int(0.7 * len(small))
    pr["splits"]["C_size_shift_le6_to_7"] = evaluate("C_size_shift_le6_to_7", T, X, groups, small[:c], small[c:], np.where(d["n_heavy"] == 7)[0])
    results["problems"]["atomization"] = pr
    T.assign(mol=d["mol"], n_heavy=d["n_heavy"]).to_csv(RES / "targets_atomization.csv", index=False)

    # ---------------- problem 2: isomerization energies (formula-grouped) ----------------
    pairs = []
    for f, g in d.groupby("formula"):
        if len(g) < 2:
            continue
        ii = g.index.to_numpy(); k = len(ii); n_pairs = min(80, k * (k - 1) // 2); seen = set()
        while len(seen) < n_pairs:
            x, y = rng.choice(k, 2, replace=False); seen.add((min(x, y), max(x, y)))
        pairs += [(f, ii[x], ii[y]) for x, y in seen]
    P = pd.DataFrame(pairs, columns=["formula", "iA", "iB"])
    EA, EB = E.iloc[P["iA"]].reset_index(drop=True), E.iloc[P["iB"]].reset_index(drop=True)
    dE = EB - EA
    T2 = targets_from(dE)
    dbc = d[bc_cols].iloc[P["iB"]].to_numpy(float) - d[bc_cols].iloc[P["iA"]].to_numpy(float)
    sbc = d[bc_cols].iloc[P["iB"]].to_numpy(float) + d[bc_cols].iloc[P["iA"]].to_numpy(float)
    comp = d[["n_C", "n_H", "n_N", "n_O", "n_S", "n_Cl", "n_heavy", "n_atom"]].iloc[P["iA"]].to_numpy(float)
    dEnn = d["Enn"].iloc[P["iB"]].to_numpy() - d["Enn"].iloc[P["iA"]].to_numpy()
    X2 = np.column_stack([dE[CHEAP_LEVELS].to_numpy(float), comp, dEnn, dbc, sbc])
    g2 = ("n_heavy=" + d["n_heavy"].iloc[P["iA"]].astype(str)).to_numpy()
    uf = P["formula"].unique(); rng.shuffle(uf); fid = P["formula"].map({f: i for i, f in enumerate(uf)}).to_numpy(); nf = len(uf)
    i_tr = np.where(fid < 0.5 * nf)[0]; i_ca = np.where((fid >= 0.5 * nf) & (fid < 0.75 * nf))[0]; i_te = np.where(fid >= 0.75 * nf)[0]
    pr2 = {"n_pairs": int(len(P)), "n_formulas": int(nf),
           "summary_kcal": {tg: {"MAE": float(T2[tg].abs().mean()), "mean": float(T2[tg].mean()), "P95_abs": float(T2[tg].abs().quantile(0.95))} for tg in TARGETS}, "splits": {}}
    pr2["splits"]["A_formula_grouped"] = evaluate("A_formula_grouped", T2, X2, g2, i_tr, i_ca, i_te)
    hetero2 = P["formula"].str.contains("S|Cl").to_numpy()
    fo = np.unique(P["formula"][~hetero2]); rng.shuffle(fo); fo_id = P["formula"].map({f: i for i, f in enumerate(fo)}).to_numpy()
    tr = np.where(fo_id < 0.7 * len(fo))[0]; ca = np.where((fo_id >= 0.7 * len(fo)) & ~np.isnan(fo_id))[0]
    pr2["splits"]["B_element_shift_to_S_Cl"] = evaluate("B_element_shift_to_S_Cl", T2, X2, g2, tr, ca, np.where(hetero2)[0])
    results["problems"]["isomerization"] = pr2
    T2.assign(formula=P["formula"], iA=P["iA"], iB=P["iB"]).to_csv(RES / "targets_isomerization.csv", index=False)

    (RES / "pilot2_results.json").write_text(json.dumps(results, indent=1))
    for pname, pr in results["problems"].items():
        print("=" * 92); print(pname, "| raw MAE:", {k: round(v["MAE"], 2) for k, v in pr["summary_kcal"].items()})
        for sname, r in pr["splits"].items():
            print(f"-- {sname}: n = {r['n_train']}/{r['n_cal']}/{r['n_test']}")
            for tg, v in r["targets"].items():
                print(f"   {tg:18s} MAE {v['MAE_raw']:7.2f} -> {v['MAE_after_model']:5.2f} | coverage {v['coverage']:.3f} width {v['mean_width']:.2f}")
            v = r["targets"]["I_bc"]
            print(f"   additive composite within 1 kcal/mol: raw {100*v['frac_raw_composite_within_1kcal']:.1f}% -> learned {100*v['frac_learned_composite_within_1kcal']:.1f}% | abstain {100*v['weighted_conformal']['frac_abstained']:.0f}%")
            for kk in ("certify_raw_composite", "certify_learned_composite", "certify_learned_composite_no_gate", "naive_rule"):
                s = v[kk]; print(f"   {kk:34s}: certified {100*s['fraction_selected']:5.1f}% FDP {s['FDP']:.3f} power {s['power']:.3f}")
    print(f"total {time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CL_ROOT}/notebooks/pilot2_figure.py
"""Figure for pilot 2 (QM7b correlation x basis non-additivity)."""
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "pilot2_qm7b"
r = json.load(open(RES / "pilot2_results.json"))
ta = pd.read_csv(RES / "targets_atomization.csv"); ti = pd.read_csv(RES / "targets_isomerization.csv")

from conformalladder.palette import BLUE, ORANGE, AQUA, YELLOW, SKY, VERM, GRAY, INK, MUTED, LIGHT_GRAY, FACE, save_fig
plt.rcParams.update({"font.size": 9, "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "semibold", "axes.titlesize": 10})
fig, ax = plt.subplots(2, 2, figsize=(11.5, 8.2), facecolor=FACE)
fig.subplots_adjust(hspace=0.5, wspace=0.3, left=0.07, right=0.98, top=0.87, bottom=0.11)

# (a) additivity error distributions
a = ax[0, 0]
a.hist(ta["I_bc"], bins=np.linspace(-5, 45, 101), color=BLUE, alpha=0.85, lw=0, label="atomization energies (7 211 molecules)")
a.hist(ti["I_bc"], bins=np.linspace(-5, 45, 101), color=ORANGE, alpha=0.85, lw=0, label=f"isomerization energies ({r['problems']['isomerization']['n_pairs']:,} pairs)".replace(",", " "))
a.axvspan(-1, 1, color=GRAY, alpha=0.18, lw=0); a.text(1.3, a.get_ylim()[1] * 0.9, "±1 kcal/mol", fontsize=8, color=MUTED)
a.set_xlabel("composite additivity error  I_bc = [CCSD(T)−MP2]$_{cc-pVDZ}$ − [CCSD(T)−MP2]$_{6-31G}$  (kcal/mol)")
a.set_ylabel("count"); a.set_title("(a) The composite assumption fails by ~22 kcal/mol for atomization,\n~1 kcal/mol for isomerization")
a.legend(frameon=False, fontsize=8, loc="upper right"); a.grid(axis="y", color="#eeede9", lw=0.6); a.set_axisbelow(True)
a.text(0.98, 0.62, f"atomization: mean {ta['I_bc'].mean():.1f}, MAE {ta['I_bc'].abs().mean():.1f}\nisomerization: MAE {ti['I_bc'].abs().mean():.2f}, within ±1: {100*(ti['I_bc'].abs()<1).mean():.0f} %",
       transform=a.transAxes, ha="right", va="top", fontsize=8, color=INK)

# (b) MAE raw vs after model, per target, atomization A / isomerization A
b = ax[0, 1]
tg = ["eps_total", "beta_basis", "kappa_correlation", "I_bc"]; names = ["total ε", "basis β\n(MP2 6-31G→cc-pVDZ)", "correlation κ\n(MP2→CCSD(T))", "interaction I_bc"]
ra = r["problems"]["atomization"]["splits"]["A_random"]["targets"]; ri = r["problems"]["isomerization"]["splits"]["A_formula_grouped"]["targets"]
x = np.arange(len(tg)); w = 0.2
b.bar(x - 1.5 * w, [ra[t]["MAE_raw"] for t in tg], w, color=GRAY, lw=0, label="atomization: raw")
b.bar(x - 0.5 * w, [ra[t]["MAE_after_model"] for t in tg], w, color=BLUE, lw=0, label="atomization: after learned layer")
b.bar(x + 0.5 * w, [ri[t]["MAE_raw"] for t in tg], w, color=LIGHT_GRAY, lw=0, label="isomerization: raw")
b.bar(x + 1.5 * w, [ri[t]["MAE_after_model"] for t in tg], w, color=ORANGE, lw=0, label="isomerization: after learned layer")
for i, t in enumerate(tg):
    b.text(x[i] - 0.5 * w, ra[t]["MAE_after_model"] + 1.5, f"{ra[t]['MAE_after_model']:.2f}", ha="center", fontsize=7, color=INK)
    b.text(x[i] + 1.5 * w, ri[t]["MAE_after_model"] + 1.5, f"{ri[t]['MAE_after_model']:.2f}", ha="center", fontsize=7, color=INK)
b.set_yscale("symlog", linthresh=2); b.set_ylim(0, 300)
b.set_xticks(x); b.set_xticklabels(names, fontsize=7.5); b.set_ylabel("MAE (kcal/mol, symlog)")
b.set_title("(b) Each layer is predictable from the cheap levels")
b.legend(frameon=False, fontsize=7.5, loc="upper right"); b.grid(axis="y", color="#eeede9", lw=0.6); b.set_axisbelow(True)

# (c) coverage of the I_bc interval per split
c = ax[1, 0]
cases = [("atomization", "A_random", "AE random"), ("atomization", "B_element_shift_to_S_Cl", "AE → S/Cl"), ("atomization", "C_size_shift_le6_to_7", "AE ≤6 → 7"),
         ("isomerization", "A_formula_grouped", "ISO formula-grouped"), ("isomerization", "B_element_shift_to_S_Cl", "ISO → S/Cl")]
x = np.arange(len(cases)); w = 0.2
for k, (t, lab, colr) in enumerate([("eps_total", "total ε", GRAY), ("beta_basis", "basis β", BLUE), ("kappa_correlation", "correlation κ", AQUA), ("I_bc", "interaction I_bc", ORANGE)]):
    cov = [r["problems"][p]["splits"][s]["targets"][t]["coverage"] for p, s, _ in cases]
    c.bar(x + (k - 1.5) * w, cov, w, color=colr, lw=0, label=lab)
abst = [r["problems"][p]["splits"][s]["targets"]["I_bc"]["weighted_conformal"]["frac_abstained"] for p, s, _ in cases]
for i, fa in enumerate(abst):
    c.text(x[i], 1.0, f"abstain {100*fa:.0f} %", ha="center", fontsize=7, color=AQUA if fa > 0.5 else MUTED, fontweight="semibold" if fa > 0.5 else "normal")
c.axhline(0.9, color=INK, lw=1, ls="--"); c.set_ylim(0, 1.35)
c.set_xticks(x); c.set_xticklabels([l for _, _, l in cases], fontsize=7.5); c.set_ylabel("empirical coverage (90 % intervals)")
c.set_title("(c) Coverage per split; under shift the router abstains")
c.legend(frameon=False, fontsize=7.5, loc="upper right", ncol=2, bbox_to_anchor=(1.0, 0.98)); c.grid(axis="y", color="#eeede9", lw=0.6); c.set_axisbelow(True)

# (d) certification of the composite
d = ax[1, 1]
keys = [("certify_raw_composite", "additive composite as is", GRAY), ("certify_learned_composite", "learned composite, BH + gate", BLUE),
        ("certify_learned_composite_no_gate", "learned composite, BH, no gate", AQUA), ("naive_rule", "naive rule", ORANGE)]
for k, (key, lab, colr) in enumerate(keys):
    vals = [r["problems"][p]["splits"][s]["targets"]["I_bc"][key] for p, s, _ in cases]
    d.bar(x + (k - 1.5) * w, [v["fraction_selected"] for v in vals], w, color=colr, lw=0, label=lab)
    for i, v in enumerate(vals):
        if v["fraction_selected"] > 0.02:
            d.text(x[i] + (k - 1.5) * w, v["fraction_selected"] + 0.015, f"{v['FDP']:.2f}", ha="center", fontsize=6.5, color=VERM if v["FDP"] > 0.12 else INK, rotation=90)
d.set_xticks(x); d.set_xticklabels([l for _, _, l in cases], fontsize=7.5); d.set_ylim(0, 1.3)
d.set_ylabel("fraction certified within 1 kcal/mol of CCSD(T)/cc-pVDZ")
d.set_title("(d) Certification (q = 0.10); number above bar = empirical FDP")
d.legend(frameon=False, fontsize=7.5, loc="upper right", ncol=2); d.grid(axis="y", color="#eeede9", lw=0.6); d.set_axisbelow(True)

fig.suptitle("ConformalLadder — pilot 2: is the composite assumption [CCSD(T)/small + MP2 basis correction] safe?\nQM7b multilevel grid, 7 211 molecules, HF/MP2/CCSD(T) × STO-3G/6-31G/cc-pVDZ",
             fontsize=10.5, fontweight="semibold", color=INK)
save_fig(fig, RES / "pilot2_summary.png"); print("saved", RES / "pilot2_summary.png")


In [ ]:
%%writefile {CL_ROOT}/notebooks/gate_comparison.py
"""Compare abstention gates for the router on the shift scenarios of pilots 1 and 2.

Gates: none | weighted-conformal (infinite weight) | k-NN support | conditional-coverage | k-NN AND conditional.
Metrics per scenario and gate: answered fraction, coverage among answered (target 0.90), certified fraction
(BH at q = 0.10 among answered) and its empirical FDP.

Run: OMP_NUM_THREADS=1 PYTHONPATH=src python notebooks/gate_comparison.py
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

from conformalladder import conformal as cp
from conformalladder import gates as G
from conformalladder import layers as L
from conformalladder.data_multixc import build as build_mx
from conformalladder.data_qm7b import build as build_qm7b
from conformalladder.data_qm9g4mp2 import BOND_KEYS

ALPHA, Q_FDR, THRESH, SEED = 0.10, 0.10, 1.0, 20260920
ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "gates"; RES.mkdir(parents=True, exist_ok=True)


def fit_mu_sigma(X, y):
    hp = dict(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=30, l2_regularization=1.0, random_state=SEED)
    mu = HistGradientBoostingRegressor(**hp).fit(X, y)
    oof = cross_val_predict(HistGradientBoostingRegressor(**hp), X, y, cv=4)
    sigma = HistGradientBoostingRegressor(**hp).fit(X, np.abs(y - oof))
    return mu, sigma


def evaluate_gates(name, X, y, i_tr, i_ca, i_te):
    mu, sg = fit_mu_sigma(X[i_tr], y[i_tr])
    m_ca, s_ca = mu.predict(X[i_ca]), np.maximum(sg.predict(X[i_ca]), 1e-3)
    m_te, s_te = mu.predict(X[i_te]), np.maximum(sg.predict(X[i_te]), 1e-3)
    sc_ca = np.abs(y[i_ca] - m_ca) / s_ca
    q = cp.conformal_quantile(sc_ca, ALPHA)
    lo, hi = m_te - q * s_te, m_te + q * s_te
    covered = (y[i_te] >= lo) & (y[i_te] <= hi)
    pv = cp.conformal_pvalues_small_error(np.abs(y[i_ca] - m_ca), s_ca, s_te, THRESH, rng=np.random.default_rng(SEED))
    sel_all = cp.benjamini_hochberg(pv, Q_FDR)
    good = np.abs(y[i_te] - m_te) < THRESH
    g_w, _ = G.weighted_conformal_gate(X[i_ca], X[i_te], ALPHA, seed=SEED)
    g_k, _ = G.knn_support_gate(X[i_ca], X[i_te], k=10, tail=0.01)
    g_c, _ = G.conditional_coverage_gate(X[i_ca], sc_ca, q, X[i_te], ALPHA, factor=2.0, seed=SEED)
    out = {"scenario": name, "n_cal": int(len(i_ca)), "n_test": int(len(i_te)), "coverage_no_gate": float(covered.mean()), "gates": {}}
    for gname, mask in [("none", np.ones(len(i_te), bool)), ("weighted_conformal", g_w), ("knn_support", g_k),
                        ("conditional_coverage", g_c), ("knn_and_conditional", g_k & g_c)]:
        n_ans = int(mask.sum())
        # BH re-run among answered items only (the certificate is issued only for answered items)
        sel = np.zeros(len(i_te), bool)
        if n_ans:
            sel[mask] = cp.benjamini_hochberg(pv[mask], Q_FDR)
        rep = cp.selection_report(sel, np.abs(y[i_te] - m_te), THRESH)
        out["gates"][gname] = {"answered": n_ans / len(i_te), "coverage_answered": float(covered[mask].mean()) if n_ans else None,
                               "certified": rep["fraction_selected"], "FDP": rep["FDP"], "power": rep["power"]}
    return out


def scenarios_pilot1(rng):
    re = pd.read_parquet(ROOT / "results" / "pilot1_multixc" / "isomer_pairs.parquet")
    XC_C, B_C = "PBE", "DZP"
    CHEAP = ["PBE", "BLYP", "BP", "PW91", "RPBE", "OLYP", "TPSS", "M06-L", "REVPBE", "HCTH/407"]
    PROPS = ["homo", "lumo", "bandgap", "mu", "alpha", "R2", "zpe", "cv"]
    feat = ([f"{XC_C}|{B_C}", "GFNXTB|none"] + [f"{f}|{B_C}" for f in CHEAP if f != XC_C] + [f"d_{p}" for p in PROPS]
            + [c for c in re.columns if c.startswith("d_") and c[2:] in BOND_KEYS + ["n_rings", "n_arom_rings", "n_rot_bonds"]]
            + [c for c in re.columns if c.startswith("s_")] + [f"n_{el}" for el in ["C", "H", "N", "O", "F"]] + ["n_heavy"])
    X = re[feat].to_numpy(float); X[:, 2:2 + len(CHEAP) - 1] -= X[:, [0]]
    sp = re[[f"{f}|{B_C}" for f in CHEAP]].to_numpy(float); X = np.column_stack([X, sp.max(1) - sp.min(1), sp.std(1)])
    budget = L.layer_budget(re, XC_C, B_C, "B3LYP(VWN5)", "TZP", disp_tag="D")
    y = budget["eps_total"].to_numpy(float)
    formulas = re["formula"].to_numpy(); uf = np.unique(formulas); rng.shuffle(uf); fid = np.array([{f: i for i, f in enumerate(uf)}[f] for f in formulas]); nf = len(uf)
    n_heavy = re["n_heavy"].to_numpy(); has_F = re["has_F"].to_numpy()
    out = [("P1 iso: formula-grouped random", X, y, np.where(fid < 0.5 * nf)[0], np.where((fid >= 0.5 * nf) & (fid < 0.75 * nf))[0], np.where(fid >= 0.75 * nf)[0])]
    small = np.where(n_heavy <= 8)[0]; big = np.where(n_heavy == 9)[0]
    fs = np.unique(formulas[small]); rng.shuffle(fs); fs_id = {f: i for i, f in enumerate(fs)}; sid = np.array([fs_id.get(f, -1) for f in formulas])
    out.append(("P1 iso: size ≤8 → 9", X, y, np.where((sid >= 0) & (sid < 0.65 * len(fs)))[0], np.where(sid >= 0.65 * len(fs))[0], rng.choice(big, size=min(20000, len(big)), replace=False)))
    noF = np.where(~has_F)[0]; fs = np.unique(formulas[noF]); rng.shuffle(fs); fs_id = {f: i for i, f in enumerate(fs)}; sid = np.array([fs_id.get(f, -1) for f in formulas])
    out.append(("P1 iso: no F → F", X, y, np.where((sid >= 0) & (sid < 0.7 * len(fs)))[0], np.where(sid >= 0.7 * len(fs))[0], np.where(has_F)[0]))
    return out


def scenarios_pilot2(rng):
    if not (ROOT / "data" / "raw" / "qm7b_multilevel" / "supplementary" / "Etot_ccpvdz.txt").exists():
        print("QM7b multilevel data not available -> pilot-2 scenarios skipped")
        return []
    d = build_qm7b()
    CHEAP = ["HF|STO-3G", "HF|6-31G", "HF|cc-pVDZ", "MP2|STO-3G", "MP2|6-31G", "CCSD(T)|STO-3G"]  # same cheap set as pilot 2
    bc_cols = [c for c in d.columns if c.startswith("bc_")]
    E = pd.DataFrame({f"{m}|{b}": d[f"AE|{m}|{b}"] for m in ["HF", "MP2", "CCSD(T)"] for b in ["STO-3G", "6-31G", "cc-pVDZ"]})
    I = (E["CCSD(T)|cc-pVDZ"] - E["MP2|cc-pVDZ"]) - (E["CCSD(T)|6-31G"] - E["MP2|6-31G"])
    X = np.column_stack([E[CHEAP].to_numpy(float), d[["n_C", "n_H", "n_N", "n_O", "n_S", "n_Cl", "n_heavy", "n_atom", "Enn"]].to_numpy(float), d[bc_cols].to_numpy(float)])
    y = I.to_numpy(float)
    idx = rng.permutation(len(d)); a, b = int(0.5 * len(d)), int(0.75 * len(d))
    out = [("P2 AE: random", X, y, idx[:a], idx[a:b], idx[b:])]
    hetero = ((d["n_S"] > 0) | (d["n_Cl"] > 0)).to_numpy(); org = rng.permutation(np.where(~hetero)[0]); c = int(0.7 * len(org))
    out.append(("P2 AE: → S/Cl", X, y, org[:c], org[c:], np.where(hetero)[0]))
    small = rng.permutation(np.where(d["n_heavy"] <= 6)[0]); c = int(0.7 * len(small))
    out.append(("P2 AE: size ≤6 → 7", X, y, small[:c], small[c:], np.where(d["n_heavy"] == 7)[0]))
    # isomerization
    pairs = []
    for f, g in d.groupby("formula"):
        if len(g) < 2:
            continue
        ii = g.index.to_numpy(); k = len(ii); n_pairs = min(80, k * (k - 1) // 2); seen = set()
        while len(seen) < n_pairs:
            x_, y_ = rng.choice(k, 2, replace=False); seen.add((min(x_, y_), max(x_, y_)))
        pairs += [(f, ii[x_], ii[y_]) for x_, y_ in seen]
    P = pd.DataFrame(pairs, columns=["formula", "iA", "iB"])
    dE = E.iloc[P["iB"]].reset_index(drop=True) - E.iloc[P["iA"]].reset_index(drop=True)
    I2 = (dE["CCSD(T)|cc-pVDZ"] - dE["MP2|cc-pVDZ"]) - (dE["CCSD(T)|6-31G"] - dE["MP2|6-31G"])
    dbc = d[bc_cols].iloc[P["iB"]].to_numpy(float) - d[bc_cols].iloc[P["iA"]].to_numpy(float)
    sbc = d[bc_cols].iloc[P["iB"]].to_numpy(float) + d[bc_cols].iloc[P["iA"]].to_numpy(float)
    comp = d[["n_C", "n_H", "n_N", "n_O", "n_S", "n_Cl", "n_heavy", "n_atom"]].iloc[P["iA"]].to_numpy(float)
    dEnn = d["Enn"].iloc[P["iB"]].to_numpy() - d["Enn"].iloc[P["iA"]].to_numpy()
    X2 = np.column_stack([dE[CHEAP].to_numpy(float), comp, dEnn, dbc, sbc]); y2 = I2.to_numpy(float)
    uf = P["formula"].unique(); rng.shuffle(uf); fid = P["formula"].map({f: i for i, f in enumerate(uf)}).to_numpy(); nf = len(uf)
    out.append(("P2 ISO: formula-grouped", X2, y2, np.where(fid < 0.5 * nf)[0], np.where((fid >= 0.5 * nf) & (fid < 0.75 * nf))[0], np.where(fid >= 0.75 * nf)[0]))
    het2 = P["formula"].str.contains("S|Cl").to_numpy(); fo = np.unique(P["formula"][~het2]); rng.shuffle(fo)
    fo_id = P["formula"].map({f: i for i, f in enumerate(fo)}).to_numpy()
    out.append(("P2 ISO: → S/Cl", X2, y2, np.where(fo_id < 0.7 * len(fo))[0], np.where((fo_id >= 0.7 * len(fo)) & ~np.isnan(fo_id))[0], np.where(het2)[0]))
    return out


def main():
    t0 = time.time(); rng = np.random.default_rng(SEED)
    scen = scenarios_pilot1(rng) + scenarios_pilot2(rng)
    results = []
    for name, X, y, tr, ca, te in scen:
        r = evaluate_gates(name, X, y, tr, ca, te); results.append(r)
        print(f"\n{name}: n_cal {r['n_cal']} n_test {r['n_test']} | coverage without gate {r['coverage_no_gate']:.3f}")
        for g, v in r["gates"].items():
            cov = f"{v['coverage_answered']:.3f}" if v["coverage_answered"] is not None else "  —  "
            print(f"   {g:22s} answered {100*v['answered']:5.1f}%  coverage {cov}  certified {100*v['certified']:5.1f}%  FDP {v['FDP']:.3f}")
    (RES / "gate_comparison.json").write_text(json.dumps(results, indent=1))
    rows = [{"scenario": r["scenario"], "gate": g, **v} for r in results for g, v in r["gates"].items()]
    pd.DataFrame(rows).to_csv(RES / "gate_comparison.csv", index=False)
    print(f"\ntotal {time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CL_ROOT}/notebooks/recalibration.py
"""Escalate-to-recalibrate: when the router detects a new domain (separability gate fires), how many
items escalated to the reference level are needed to restore valid coverage and FDR there?

For each harmful shift scenario we draw n_new labelled items from the shifted test set and use them
(a) only as the calibration set, or (b) half added to training, half as calibration; the remaining
shifted items are evaluated (coverage of 90 % intervals, certified fraction, FDP).  Averaged over draws.

Run: OMP_NUM_THREADS=1 PYTHONPATH=src python notebooks/recalibration.py
"""
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
from gate_comparison import fit_mu_sigma, scenarios_pilot1, scenarios_pilot2  # noqa: E402
from conformalladder import conformal as cp  # noqa: E402

ALPHA, Q_FDR, THRESH, SEED = 0.10, 0.10, 1.0, 20260920
ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "gates"; RES.mkdir(parents=True, exist_ok=True)
N_NEW = [25, 50, 100, 200, 400]
N_DRAWS = 3


def run(name, X, y, i_tr, i_te, rng):
    rows = []
    mu0, sg0 = fit_mu_sigma(X[i_tr], y[i_tr])
    for n_new in N_NEW:
        if n_new * 2 > len(i_te):
            continue
        for draw in range(N_DRAWS):
            perm = rng.permutation(i_te); new, rest = perm[:n_new], perm[n_new:]
            # (a) calibrate only
            for variant, (mu, sg, cal) in {"calibrate_only": (mu0, sg0, new)}.items():
                m_ca, s_ca = mu.predict(X[cal]), np.maximum(sg.predict(X[cal]), 1e-3)
                m_te, s_te = mu.predict(X[rest]), np.maximum(sg.predict(X[rest]), 1e-3)
                lo, hi, q = cp.scaled_residual_intervals(m_ca, s_ca, y[cal], m_te, s_te, ALPHA)
                pv = cp.conformal_pvalues_small_error(np.abs(y[cal] - m_ca), s_ca, s_te, THRESH, rng=np.random.default_rng(SEED + draw))
                sel = cp.benjamini_hochberg(pv, Q_FDR); rep = cp.selection_report(sel, np.abs(y[rest] - m_te), THRESH)
                rows.append({"scenario": name, "variant": variant, "n_new": n_new, "draw": draw, "coverage": cp.coverage(y[rest], lo, hi),
                             "mean_width": float(np.mean(hi - lo)), "MAE_after_model": float(np.mean(np.abs(y[rest] - m_te))),
                             "certified": rep["fraction_selected"], "FDP": rep["FDP"]})
            # (b) half to training, half to calibration
            if n_new in (100, 400):
                h = n_new // 2; tr2 = np.r_[i_tr, new[:h]]; cal = new[h:]
                mu, sg = fit_mu_sigma(X[tr2], y[tr2])
                m_ca, s_ca = mu.predict(X[cal]), np.maximum(sg.predict(X[cal]), 1e-3)
                m_te, s_te = mu.predict(X[rest]), np.maximum(sg.predict(X[rest]), 1e-3)
                lo, hi, q = cp.scaled_residual_intervals(m_ca, s_ca, y[cal], m_te, s_te, ALPHA)
                pv = cp.conformal_pvalues_small_error(np.abs(y[cal] - m_ca), s_ca, s_te, THRESH, rng=np.random.default_rng(SEED + draw))
                sel = cp.benjamini_hochberg(pv, Q_FDR); rep = cp.selection_report(sel, np.abs(y[rest] - m_te), THRESH)
                rows.append({"scenario": name, "variant": "retrain_half_calibrate_half", "n_new": n_new, "draw": draw, "coverage": cp.coverage(y[rest], lo, hi),
                             "mean_width": float(np.mean(hi - lo)), "MAE_after_model": float(np.mean(np.abs(y[rest] - m_te))),
                             "certified": rep["fraction_selected"], "FDP": rep["FDP"]})
    return rows


def main():
    t0 = time.time(); rng = np.random.default_rng(SEED)
    scen = {s[0]: s for s in scenarios_pilot1(rng) + scenarios_pilot2(rng)}
    harmful = [n for n in ["P1 iso: size ≤8 → 9", "P2 AE: → S/Cl", "P2 AE: size ≤6 → 7", "P2 ISO: → S/Cl"] if n in scen]
    rows = []
    for name in harmful:
        _, X, y, tr, ca, te = scen[name]
        rows += run(name, X, y, tr, te, rng)
        print(name, "done", f"{time.time()-t0:.0f}s")
    df = pd.DataFrame(rows); df.to_csv(RES / "recalibration.csv", index=False)
    summ = df.groupby(["scenario", "variant", "n_new"]).agg(coverage=("coverage", "mean"), coverage_sd=("coverage", "std"), width=("mean_width", "mean"),
                                                            MAE=("MAE_after_model", "mean"), certified=("certified", "mean"), FDP=("FDP", "mean"), FDP_max=("FDP", "max")).reset_index()
    summ.to_csv(RES / "recalibration_summary.csv", index=False)
    print(summ.round(3).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CL_ROOT}/notebooks/gates_figure.py
"""Figure: abstention gates and the escalate-to-recalibrate loop (reads results/gates/)."""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "gates"
from conformalladder.palette import BLUE, ORANGE, AQUA, YELLOW, SKY, VERM, GRAY, INK, MUTED, LIGHT_GRAY, FACE, save_fig
plt.rcParams.update({"font.size": 9, "axes.edgecolor": GRAY, "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "semibold", "axes.titlesize": 9.5})
s = pd.read_csv(RES / "recalibration_summary.csv"); g = pd.read_csv(RES / "gate_comparison.csv")
harm = ["P1 iso: size ≤8 → 9", "P2 AE: → S/Cl", "P2 AE: size ≤6 → 7", "P2 ISO: → S/Cl"]; ben = ["P1 iso: no F → F"]
gates = ["none", "weighted_conformal", "knn_support", "conditional_coverage", "knn_and_conditional"]; cols = [GRAY, BLUE, ORANGE, AQUA, YELLOW]
fig, ax = plt.subplots(1, 3, figsize=(13.5, 4.4), facecolor=FACE); fig.subplots_adjust(wspace=0.3, left=0.055, right=0.985, top=0.8, bottom=0.19)
a = ax[0]
for gname, colr in zip(gates, cols):
    d = g[(g.gate == gname) & (g.scenario.isin(harm))]; a.scatter(d["answered"], d["FDP"], s=45, color=colr, label=gname.replace("_", " "), lw=0, alpha=0.9)
    d2 = g[(g.gate == gname) & (g.scenario.isin(ben))]; a.scatter(d2["answered"], d2["FDP"], s=45, color=colr, marker="^", lw=0, alpha=0.9)
a.axhline(0.10, color=INK, lw=1, ls="--"); a.text(0.02, 0.115, "q = 0.10", fontsize=8)
a.set_xlabel("fraction of shifted items answered"); a.set_ylabel("empirical FDP of the certificate")
a.set_title("(a) Gates: harmful shifts (●), benign shift (▲)"); a.legend(frameon=False, fontsize=7.5, loc="lower center", bbox_to_anchor=(0.5, 0.13)); a.grid(color="#eeede9", lw=0.6); a.set_axisbelow(True)
a.text(0.1, 0.46, "the separability gate abstains on\n99–100 % of every harmful batch;\nthe other gates certify almost\nas if there were no gate", fontsize=8, color=BLUE, va="bottom")
b = ax[1]
mk = {k: v for k, v in {"P1 iso: size ≤8 → 9": (BLUE, "P1 iso ≤8→9"), "P2 AE: → S/Cl": (ORANGE, "P2 AE →S/Cl"), "P2 AE: size ≤6 → 7": (AQUA, "P2 AE ≤6→7"),
                        "P2 ISO: → S/Cl": (YELLOW, "P2 ISO →S/Cl")}.items() if k in set(s["scenario"])}
for sc, (colr, lab) in mk.items():
    d = s[(s.scenario == sc) & (s.variant == "calibrate_only")].sort_values("n_new")
    b.errorbar(d["n_new"], d["coverage"], yerr=d["coverage_sd"], color=colr, marker="o", ms=5, lw=1.8, capsize=2, label=lab)
    d = s[(s.scenario == sc) & (s.variant != "calibrate_only")].sort_values("n_new"); b.plot(d["n_new"], d["coverage"], color=colr, marker="s", ms=6, lw=0, mfc="none", mew=1.5)
b.axhline(0.9, color=INK, lw=1, ls="--"); b.set_xscale("log"); b.set_xticks([25, 50, 100, 200, 400]); b.set_xticklabels([25, 50, 100, 200, 400])
b.set_ylim(0.6, 1.0); b.set_xlabel("items escalated to the reference level (calibration)"); b.set_ylabel("coverage on the remaining shifted items")
b.set_title("(b) Escalate-to-recalibrate: coverage back with ~50 items"); b.legend(frameon=False, fontsize=7.5, loc="lower right", bbox_to_anchor=(1.0, 0.08)); b.grid(color="#eeede9", lw=0.6); b.set_axisbelow(True)
b.text(26, 0.63, "○ calibrate only   □ half retrain, half calibrate", fontsize=7.5, color=MUTED)
c = ax[2]
for sc, (colr, lab) in mk.items():
    d = s[(s.scenario == sc) & (s.variant == "calibrate_only")].sort_values("n_new"); c.plot(d["n_new"], d["width"] / 2, color=colr, marker="o", ms=5, lw=1.8, label=lab)
    d = s[(s.scenario == sc) & (s.variant != "calibrate_only")].sort_values("n_new"); c.plot(d["n_new"], d["width"] / 2, color=colr, marker="s", ms=6, lw=0, mfc="none", mew=1.5)
c.set_yscale("log"); c.set_xscale("log"); c.set_xticks([25, 50, 100, 200, 400]); c.set_xticklabels([25, 50, 100, 200, 400])
c.set_xlabel("items escalated"); c.set_ylabel("mean 90 % half-width (kcal/mol, log)"); c.set_title("(c) Sharpness needs retraining")
c.axhline(1.0, color=INK, lw=1, ls="--"); c.text(26, 1.15, "1 kcal/mol", fontsize=8); c.grid(color="#eeede9", lw=0.6); c.set_axisbelow(True)
fig.suptitle("ConformalLadder — routing under distribution shift: detection gates and the escalate-to-recalibrate loop", fontsize=10.5, fontweight="semibold")
save_fig(fig, RES / "gates_and_recalibration.png"); print("saved", RES / "gates_and_recalibration.png")


In [ ]:
%%writefile {CL_ROOT}/notebooks/w4_17_budget.py
"""First multi-layer error budget on W4-17 (curated subsets: TAE, BDE99, HAT707, ISO20, SN13).

Cheap level  : xc_c / b_c   (default PBE / def2-SVP)
Big basis    : b_big        (default def2-TZVP)
Best DFT     : chosen on the training split among the functionals available at b_big (+D3BJ)
Layers       : beta (basis), phi (functional), delta (dispersion), rho (remainder to W4 reference)
Interaction  : I_bf (non-additivity of basis and functional corrections)

For every layer and for the total we fit a point model + dispersion model on cheap features
and calibrate scaled split-conformal intervals; the total error is also certified
(|eps - eps_hat| < 1 kcal/mol) with conformal p-values + BH.

Run: PYTHONPATH=src python notebooks/w4_17_budget.py [--cheap PBE def2-SVP] [--big def2-TZVP]
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

from conformalladder import conformal as cp
from conformalladder import layers as L

ALPHA, Q_FDR, THRESH, SEED = 0.10, 0.10, 1.0, 20260920
N_REPEATS = 10   # random single-reference splits for the repeated-split statistics (seed + k)
ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "w4-17"
RES.mkdir(parents=True, exist_ok=True)
LAYERS = ["eps_total", "beta_basis", "phi_functional", "delta_dispersion", "rho_remainder", "I_bf"]


def fit_mu_sigma(X, y, seed=SEED):
    hp = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=10,
              l2_regularization=1.0, random_state=seed)
    mu = HistGradientBoostingRegressor(**hp).fit(X, y)
    oof = cross_val_predict(HistGradientBoostingRegressor(**hp), X, y, cv=5)
    sigma = HistGradientBoostingRegressor(**hp).fit(X, np.abs(y - oof))
    return mu, sigma


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cheap", nargs=2, default=["PBE", "def2-SVP"])
    ap.add_argument("--big", default="def2-TZVP")
    ap.add_argument("--grid", default=str(ROOT / "data" / "processed" / "w4-17_grid.jsonl"))
    args = ap.parse_args()
    xc_c, b_c = args.cheap; b_big = args.big

    reactions = L.load_reactions("W4-17", include_re=False)
    dft, disp, fod = L.load_grid(args.grid)
    re = L.reaction_energies(reactions, dft, disp)
    feats = L.species_features(reactions, dft, fod, xc_c, b_c)
    avail = sorted({c.split("|")[0] for c in re.columns if "|" in c and c.endswith(b_big) and re[c].notna().mean() > 0.95})
    avail_small = sorted({c.split("|")[0] for c in re.columns if "|" in c and c.endswith(b_c) and re[c].notna().mean() > 0.95})
    print("functionals complete at", b_big, ":", avail, "| at", b_c, ":", avail_small)

    # ---- splits over reactions: random (exchangeable) + OOD by multireference species ------------
    rng = np.random.default_rng(SEED)
    mr_species = set((L.DB / "W4-17" / "W4-17-MR.list").read_text().split())
    is_mr = np.array([any(s in mr_species for s in st) for st in reactions["stoich"]])
    idx = np.arange(len(re))
    perm = rng.permutation(idx[~is_mr])
    n = len(perm); i_tr, i_ca, i_te = perm[: int(0.5 * n)], perm[int(0.5 * n): int(0.75 * n)], perm[int(0.75 * n):]
    i_mr = idx[is_mr]

    # ---- choose the "best" DFT level on the training split (MAE vs reference, +D3BJ if available) ---
    def disp_tag_for(x):
        """first dispersion correction with parameters for functional x: D3(BJ) > D3(zero) > D4"""
        for tag in ("D3BJ", "D3ZERO", "D4"):
            col = f"disp:{x}+{tag}"
            if col in re and re[col].notna().all():
                return tag
        return None
    cand, tags = {}, {}
    for x in avail:
        if x == xc_c:
            continue
        tag = disp_tag_for(x); tags[x] = tag
        vals = re[f"{x}|{b_big}"] + (re[f"disp:{x}+{tag}"] if tag else 0.0)
        cand[x] = float(np.mean(np.abs(re["ref"].values[i_tr] - vals.values[i_tr])))
    if not cand:
        raise SystemExit("need at least one functional besides the cheap one at the big basis")
    xc_best = min(cand, key=cand.get); disp_tag = tags[xc_best] or "D3BJ"
    print("training-split MAE at", b_big, "+dispersion:", {k: (round(v, 2), tags[k]) for k, v in cand.items()}, "-> best:", xc_best, "with", disp_tag)

    budget = L.layer_budget(re, xc_c, b_c, xc_best, b_big, disp_tag=disp_tag)
    assert np.nanmax(np.abs(budget["check_identity"])) < 1e-8
    summary = L.summarize_budget(budget)
    summary.to_csv(RES / f"budget_summary_{xc_c}_{b_c}.csv", index=False)
    budget.to_csv(RES / f"budget_{xc_c}_{b_c}.csv", index=False)

    # ---- features available from the cheap level only ---------------------------------------
    F = feats.copy()
    F["dE_cheap"] = re[f"{xc_c}|{b_c}"]
    F["disp_cheap"] = re.get(f"disp:{xc_c}+D3BJ", np.nan)
    spread_cols = [f"{x}|{b_c}" for x in avail_small if x != xc_c]
    if spread_cols:   # multi-functional disagreement in the *small* basis (cheap)
        F["spread_small_basis"] = re[spread_cols].max(axis=1) - re[spread_cols].min(axis=1)
        F["mean_shift_small_basis"] = re[spread_cols].mean(axis=1) - re[f"{xc_c}|{b_c}"]
    F["is_mr"] = is_mr.astype(float)
    feature_cols = [c for c in F.columns if c not in ("rid", "is_mr")]
    X = F[feature_cols].to_numpy(float)

    def evaluate(i_tr, i_ca, i_te, i_mr, seed):
        """Fit every layer on i_tr, calibrate on i_ca, evaluate on i_te (+ the multireference set i_mr)."""
        out = {}
        for layer in LAYERS:
            y = budget[layer].to_numpy(float)
            ok = np.isfinite(y) & np.isfinite(X).all(axis=1)
            tr = i_tr[ok[i_tr]]; ca = i_ca[ok[i_ca]]; te = i_te[ok[i_te]]; mr = i_mr[ok[i_mr]]
            if len(tr) < 50 or len(ca) < 20 or len(te) < 20:
                continue
            mu, sg = fit_mu_sigma(X[tr], y[tr], seed=seed)
            m_ca, s_ca = mu.predict(X[ca]), np.maximum(sg.predict(X[ca]), 1e-3)
            m_te, s_te = mu.predict(X[te]), np.maximum(sg.predict(X[te]), 1e-3)
            lo, hi, q = cp.scaled_residual_intervals(m_ca, s_ca, y[ca], m_te, s_te, ALPHA)
            rec = {"MAE_raw": float(np.mean(np.abs(y[te]))), "MAE_after_model": float(np.mean(np.abs(y[te] - m_te))),
                   "coverage": cp.coverage(y[te], lo, hi), "mean_width": float(np.mean(hi - lo)), "q": float(q),
                   "by_subset": cp.group_coverage(y[te], lo, hi, budget["subset"].to_numpy()[te])}
            if len(mr) >= 5:   # OOD: reactions involving multireference species
                m_mr, s_mr = mu.predict(X[mr]), np.maximum(sg.predict(X[mr]), 1e-3)
                lo_m, hi_m = m_mr - q * s_mr, m_mr + q * s_mr
                rec["OOD_multireference"] = {"n": int(len(mr)), "coverage": cp.coverage(y[mr], lo_m, hi_m),
                                             "MAE_raw": float(np.mean(np.abs(y[mr]))), "MAE_after_model": float(np.mean(np.abs(y[mr] - m_mr)))}
            if layer == "eps_total":
                pv = cp.conformal_pvalues_small_error(np.abs(y[ca] - m_ca), s_ca, s_te, THRESH, rng=np.random.default_rng(seed))
                sel = cp.benjamini_hochberg(pv, Q_FDR)
                rec["certification_BH"] = cp.selection_report(sel, np.abs(y[te] - m_te), THRESH)
                rec["frac_raw_within_1kcal"] = float(np.mean(np.abs(y[te]) < THRESH))
            out[layer] = rec
        return out

    results = {"cheap": [xc_c, b_c], "big_basis": b_big, "best_functional": xc_best, "disp_tag": disp_tag, "train_MAE_candidates": cand,
               "n_reactions": int(len(re)), "n_train": len(i_tr), "n_cal": len(i_ca), "n_test": len(i_te), "n_mr": len(i_mr),
               "features": feature_cols, "layers": evaluate(i_tr, i_ca, i_te, i_mr, SEED)}

    # ---- repeated random splits (same target rung): mean and s.d. of every headline metric -------------
    metrics = ["MAE_raw", "MAE_after_model", "coverage", "mean_width", "OOD_coverage", "OOD_MAE_after_model"]
    acc = {layer: {m: [] for m in metrics} for layer in LAYERS}; cert = {"fraction_selected": [], "FDP": [], "base_rate_good": []}
    for k in range(N_REPEATS):
        seed_k = SEED + k
        perm_k = np.random.default_rng(seed_k).permutation(idx[~is_mr])
        tr_k, ca_k, te_k = perm_k[: int(0.5 * n)], perm_k[int(0.5 * n): int(0.75 * n)], perm_k[int(0.75 * n):]
        lay_k = results["layers"] if k == 0 else evaluate(tr_k, ca_k, te_k, i_mr, seed_k)
        for layer, rec in lay_k.items():
            for m in ("MAE_raw", "MAE_after_model", "coverage", "mean_width"):
                acc[layer][m].append(rec[m])
            if "OOD_multireference" in rec:
                acc[layer]["OOD_coverage"].append(rec["OOD_multireference"]["coverage"]); acc[layer]["OOD_MAE_after_model"].append(rec["OOD_multireference"]["MAE_after_model"])
        c = lay_k["eps_total"]["certification_BH"]
        for m in cert:
            cert[m].append(c[m])
        print(f"repeat {k + 1}/{N_REPEATS} (seed {seed_k}): eps coverage {lay_k['eps_total']['coverage']:.3f}, MAE {lay_k['eps_total']['MAE_after_model']:.2f}")
    results["repeats"] = {"n_splits": N_REPEATS, "seeds": [SEED + k for k in range(N_REPEATS)],
                          "layers": {layer: {m: {"mean": float(np.mean(v)), "sd": float(np.std(v, ddof=1))} for m, v in d.items() if v} for layer, d in acc.items()},
                          "certification_BH": {m: {"mean": float(np.mean(v)), "sd": float(np.std(v, ddof=1))} for m, v in cert.items()}}
    (RES / f"results_{xc_c}_{b_c}.json").write_text(json.dumps(results, indent=1))
    print(summary[["subset", "n", "eps_total_MAE", "beta_basis_MAE", "phi_functional_MAE", "delta_dispersion_MAE", "rho_remainder_MAE", "I_bf_MAE", "eps_after_ladder_MAE"]].round(2).to_string(index=False))
    for k, v in results["layers"].items():
        print(f"{k:18s} MAE {v['MAE_raw']:6.2f} -> {v['MAE_after_model']:5.2f} | coverage {v['coverage']:.3f} width {v['mean_width']:.2f}"
              + (f" | MR-OOD coverage {v['OOD_multireference']['coverage']:.3f}" if "OOD_multireference" in v else "") + "   [primary split]")
    print(f"mean ± s.d. over {N_REPEATS} random splits:")
    for k, v in results["repeats"]["layers"].items():
        print(f"{k:18s} MAE {v['MAE_raw']['mean']:6.2f} -> {v['MAE_after_model']['mean']:5.2f} ± {v['MAE_after_model']['sd']:.2f} | coverage {v['coverage']['mean']:.3f} ± {v['coverage']['sd']:.3f}"
              f" width {v['mean_width']['mean']:.2f}" + (f" | MR-OOD coverage {v['OOD_coverage']['mean']:.3f} ± {v['OOD_coverage']['sd']:.3f}" if "OOD_coverage" in v else ""))
    if "eps_total" in results["layers"]:
        print("certification (primary split):", results["layers"]["eps_total"]["certification_BH"])
        print("certification (mean over splits):", {m: round(v["mean"], 3) for m, v in results["repeats"]["certification_BH"].items()})


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CL_ROOT}/notebooks/w4_17_figure.py
"""Figure for the W4-17 layer budget (reads results/w4-17/)."""
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "w4-17"
tag = sys.argv[1] if len(sys.argv) > 1 else "PBE_def2-SVP"
r = json.load(open(RES / f"results_{tag}.json"))
b = pd.read_csv(RES / f"budget_{tag}.csv")
s = pd.read_csv(RES / f"budget_summary_{tag}.csv")

from conformalladder.palette import BLUE, ORANGE, AQUA, YELLOW, SKY, VERM, GRAY, INK, MUTED, LIGHT_GRAY, FACE, save_fig
plt.rcParams.update({"font.size": 9, "axes.edgecolor": GRAY, "axes.labelcolor": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "semibold", "axes.titlesize": 10})
fig, ax = plt.subplots(2, 2, figsize=(11.5, 8), facecolor=FACE)
fig.subplots_adjust(hspace=0.5, wspace=0.3, left=0.07, right=0.98, top=0.88, bottom=0.1)
xc_c, b_c = r["cheap"]; xc_b, b_big = r["best_functional"], r["big_basis"]
disp_name = {"D3BJ": "D3(BJ)", "D3ZERO": "D3(0)", "D4": "D4"}.get(r.get("disp_tag", "D3BJ"), r.get("disp_tag", "D3BJ"))
short = {"TAE_W4-17": "TAE (203)", "BDE99": "BDE99", "HAT707": "HAT707", "ISOMERIZATION20": "ISO20", "SN13": "SN13", "ALL": "ALL (1042)"}

# (a) layer MAE per subset (grouped bars) + total
a = ax[0, 0]
subs = [x for x in ["TAE_W4-17", "BDE99", "HAT707", "ISOMERIZATION20", "SN13", "ALL"] if x in set(s["subset"])]
ss = s.set_index("subset").loc[subs]
layers = [("beta_basis_MAE", f"basis  {b_c}→{b_big}", BLUE), ("phi_functional_MAE", f"functional  {xc_c}→{xc_b}", ORANGE),
          ("delta_dispersion_MAE", f"dispersion  {disp_name}", AQUA), ("rho_remainder_MAE", "remainder to W4 reference", YELLOW)]
x = np.arange(len(subs)); w = 0.18
for k, (col, lab, colr) in enumerate(layers):
    a.bar(x + (k - 1.5) * w, ss[col], w, color=colr, lw=0, label=lab)
a.plot(x, ss["eps_total_MAE"], "o", color=INK, ms=5, label=f"total error of {xc_c}/{b_c}")
a.plot(x, ss["I_bf_MAE"], "s", color=GRAY, ms=5, label="non-additivity |I_bf|")
a.set_xticks(x); a.set_xticklabels([short[k] for k in subs], fontsize=8)
a.set_ylabel("MAE (kcal/mol)"); a.set_title("(a) Layer-resolved error budget per subset")
a.legend(frameon=False, fontsize=7.5, ncol=2, loc="upper right"); a.grid(axis="y", color="#eeede9", lw=0.6); a.set_axisbelow(True)

# (b) non-additivity vs functional correction (per reaction)
bb = ax[0, 1]
bb.scatter(b["phi_functional"], b["I_bf"], s=8, color=BLUE, alpha=0.5, lw=0)
bb.axhline(0, color=GRAY, lw=0.8)
lim = np.nanpercentile(np.abs(b["phi_functional"]), 99)
bb.set_xlim(-lim, lim)
bb.set_xlabel(f"functional correction φ = ΔE({xc_b}/{b_big}) − ΔE({xc_c}/{b_big}) (kcal/mol)")
bb.set_ylabel("interaction I_bf (kcal/mol)")
rel = float(np.nanmedian(np.abs(b["I_bf"]) / np.maximum(np.abs(b["phi_functional"]), 1e-6)))
bb.set_title("(b) Non-additivity of basis and functional corrections")
bb.text(0.02, 0.95, f"median |I_bf| = {np.nanmedian(np.abs(b['I_bf'])):.2f} kcal/mol\nmedian |I_bf|/|φ| = {rel:.2f}\n"
        f"|I_bf| > 1 kcal/mol in {100*np.nanmean(np.abs(b['I_bf'])>1):.0f} % of reactions", transform=bb.transAxes, va="top", fontsize=8, color=INK)
bb.grid(color="#eeede9", lw=0.6); bb.set_axisbelow(True)

# (c) coverage per layer: random test vs multireference OOD
c = ax[1, 0]
lay = [k for k in ["eps_total", "beta_basis", "phi_functional", "delta_dispersion", "rho_remainder", "I_bf"] if k in r["layers"]]
names = {"eps_total": "total ε", "beta_basis": "basis β", "phi_functional": "functional φ", "delta_dispersion": "dispersion δ", "rho_remainder": "remainder ρ", "I_bf": "interaction"}
REP = r.get("repeats", {}).get("layers", {})   # mean over repeated random splits (falls back to the primary split)
def stat(k, m, m_rep=None):
    m_rep = m_rep or m
    if k in REP and m_rep in REP[k]:
        return REP[k][m_rep]["mean"]
    return r["layers"][k][m] if m in r["layers"][k] else r["layers"][k].get("OOD_multireference", {}).get("coverage", np.nan)
cov = [stat(k, "coverage") for k in lay]
cov_mr = [stat(k, "OOD_coverage") if k in REP else r["layers"][k].get("OOD_multireference", {}).get("coverage", np.nan) for k in lay]
n_rep = r.get("repeats", {}).get("n_splits", 1)
x = np.arange(len(lay)); w = 0.36
c.bar(x - w / 2, cov, w, color=BLUE, lw=0, label="random test reactions")
c.bar(x + w / 2, cov_mr, w, color=ORANGE, lw=0, label="reactions with multireference species (OOD)")
for i, v in enumerate(cov):
    c.text(x[i] - w / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=7, color=MUTED)
for i, v in enumerate(cov_mr):
    if np.isfinite(v): c.text(x[i] + w / 2, v + 0.01, f"{v:.2f}", ha="center", fontsize=7, color=MUTED)
c.axhline(0.9, color=INK, lw=1, ls="--", label="target 0.90")
c.set_xticks(x); c.set_xticklabels([names[k] for k in lay], fontsize=7.5); c.set_ylim(0, 1.3)
c.set_ylabel("empirical coverage (90 % intervals)"); c.set_title(f"(c) Coverage per layer (mean over {n_rep} random splits)" if n_rep > 1 else "(c) Coverage per layer")
c.legend(frameon=False, fontsize=7.5, loc="upper left"); c.grid(axis="y", color="#eeede9", lw=0.6); c.set_axisbelow(True)

# (d) MAE before/after model and interval width per layer
d = ax[1, 1]
raw = [stat(k, "MAE_raw") for k in lay]; after = [stat(k, "MAE_after_model") for k in lay]; wid = [stat(k, "mean_width") / 2 for k in lay]
d.bar(x - w / 2, raw, w, color=GRAY, lw=0, label="MAE of the layer (no model)")
d.bar(x + w / 2, after, w, color=BLUE, lw=0, label="MAE after learned prediction")
d.plot(x + w / 2, wid, "v", color=ORANGE, ms=6, label="mean 90 % half-width")
d.set_xticks(x); d.set_xticklabels([names[k] for k in lay], fontsize=7.5)
d.set_ylabel("kcal/mol"); d.set_title(f"(d) Predictability of each layer (mean over {n_rep} random splits)" if n_rep > 1 else "(d) Predictability of each layer from the cheap calculation")
d.legend(frameon=False, fontsize=7.5, loc="upper right"); d.grid(axis="y", color="#eeede9", lw=0.6); d.set_axisbelow(True)

cert = r["layers"].get("eps_total", {}).get("certification_BH", {})
fig.suptitle(f"ConformalLadder on W4-17 (1042 relative energies, W4 references)\ncheap level {xc_c}/{b_c} → ladder {xc_b}-{disp_name}/{b_big} → reference;  "
             f"certified at q = 0.10: {100*cert.get('fraction_selected', 0):.0f} % of test reactions (FDP {cert.get('FDP', float('nan')):.2f})",
             fontsize=10.5, fontweight="semibold", color=INK)
save_fig(fig, RES / f"w4_17_budget_{tag}.png")
print("saved", RES / f"w4_17_budget_{tag}.png")


In [ ]:
%%writefile {CL_ROOT}/notebooks/w4_17_rungs.py
"""W4-17: the layer budget for every choice of the DFT rung (no ML), FOD validation, and figure.

Run: PYTHONPATH=src python notebooks/w4_17_rungs.py
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

from conformalladder import layers as L

ROOT = Path(__file__).resolve().parents[1]
RES = ROOT / "results" / "w4-17"; RES.mkdir(parents=True, exist_ok=True)
GRID = ROOT / "data" / "processed" / "w4-17_grid.jsonl"
XC_C, B_C, B_BIG = "PBE", "def2-SVP", "def2-TZVP"
from conformalladder.palette import BLUE, ORANGE, AQUA, YELLOW, SKY, VERM, GRAY, INK, MUTED, LIGHT_GRAY, FACE, save_fig


def main():
    reactions = L.load_reactions("W4-17"); dft, disp, fod = L.load_grid(GRID)
    re = L.reaction_energies(reactions, dft, disp)
    mr = set((L.DB / "W4-17" / "W4-17-MR.list").read_text().split())
    is_mr = np.array([any(s in mr for s in st) for st in reactions["stoich"]])
    avail = sorted({c.split("|")[0] for c in re.columns if "|" in c and c.endswith(B_BIG) and re[c].notna().mean() > 0.95})
    rungs = [x for x in ["BLYP", "TPSS", "r2SCAN", "B3LYP", "PBE0", "wB97X", "M06-2X"] if x in avail and x != XC_C]
    rows, tags = [], {}
    for xc in rungs:
        tag = next((t for t in ("D3BJ", "D3ZERO", "D4") if f"disp:{xc}+{t}" in re and re[f"disp:{xc}+{t}"].notna().all()), "D3BJ"); tags[xc] = tag
        b = L.layer_budget(re, XC_C, B_C, xc, B_BIG, disp_tag=tag)
        for sub, g in [("ALL", b), ("nonMR", b[~is_mr]), ("MR", b[is_mr])]:
            rows.append({"rung": f"{xc}-{tag}", "subset": sub, "n": int(len(g)), "eps_cheap": g.eps_total.abs().mean(), "beta": g.beta_basis.abs().mean(),
                         "phi": g.phi_functional.abs().mean(), "delta": g.delta_dispersion.abs().mean(), "rho_after_ladder": g.rho_remainder.abs().mean(),
                         "I_bf": g.I_bf.abs().mean(), "I_bf_gt1_pct": 100 * (g.I_bf.abs() > 1).mean(), "I_fd": g.I_fd.abs().mean()})
    t = pd.DataFrame(rows); t.to_csv(RES / "rung_comparison.csv", index=False)
    print("Layer budget per rung (MAE, kcal/mol), cheap level", f"{XC_C}/{B_C}:")
    print(t[t.subset == "ALL"].round(2).to_string(index=False))

    # cheap diagnostics
    fx = [x for x in avail]
    sp_svp = re[[f"{f}|{B_C}" for f in fx]].std(axis=1)
    bref = L.layer_budget(re, XC_C, B_C, "B3LYP" if "B3LYP" in rungs else rungs[0], B_BIG)
    diag = {"n_functionals": len(fx), "spearman_spread_svp_vs_abs_rho": float(spearmanr(sp_svp, bref.rho_remainder.abs())[0]),
            "spearman_spread_svp_vs_abs_Ibf": float(spearmanr(sp_svp, bref.I_bf.abs())[0])}
    if len(fod):
        f = fod.set_index("species"); mols = f[f["natom"] > 1]
        diag["fod_auc_for_MR_membership_molecules"] = float(roc_auc_score(mols.index.isin(mr), mols["n_fod"]))
        diag["fod_converged_fraction"] = float(f["converged"].mean())
    (RES / "diagnostics.json").write_text(json.dumps(diag, indent=1)); print("diagnostics:", diag)

    # figure
    plt.rcParams.update({"font.size": 9, "axes.edgecolor": GRAY, "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "semibold", "axes.titlesize": 9.5})
    order = [f"{x}-{tags[x]}" for x in rungs]
    lab = [r.replace("-D3BJ", "\n-D3(BJ)").replace("-D3ZERO", "\n-D3(0)").replace("-D4", "\n-D4").replace("wB97X", "ωB97X").replace("r2SCAN", "r²SCAN") for r in order]
    fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.6), facecolor=FACE); fig.subplots_adjust(wspace=0.26, left=0.06, right=0.99, top=0.8, bottom=0.2)
    a = ax[0]; d = t[t.subset == "ALL"].set_index("rung").loc[order]
    x = np.arange(len(order)); w = 0.2
    a.bar(x - 1.5 * w, d["beta"], w, color=BLUE, lw=0, label=f"basis β ({B_C}→{B_BIG}, {XC_C})")
    a.bar(x - 0.5 * w, d["phi"], w, color=ORANGE, lw=0, label=f"functional φ ({XC_C}→rung)")
    a.bar(x + 0.5 * w, d["delta"], w, color=AQUA, lw=0, label="dispersion δ")
    a.bar(x + 1.5 * w, d["rho_after_ladder"], w, color=YELLOW, lw=0, label="remainder ρ to W4 (= error after the ladder)")
    a.plot(x, d["I_bf"], "s", color=INK, ms=6, label="non-additivity |I_bf| (basis × functional)")
    a.set_xticks(x); a.set_xticklabels(lab, fontsize=7.5); a.set_ylabel(f"MAE over {len(re)} W4-17 relative energies (kcal/mol)")
    a.set_title(f"(a) One cheap calculation ({XC_C}/{B_C}), {len(rungs)} choices of the DFT rung"); a.legend(frameon=False, fontsize=7.5, loc="upper left")
    a.grid(axis="y", color="#eeede9", lw=0.6); a.set_axisbelow(True); a.set_ylim(0, max(13, d["phi"].max() * 1.25))
    b = ax[1]
    nm = t[t.subset == "nonMR"].set_index("rung").loc[order]; mrr = t[t.subset == "MR"].set_index("rung").loc[order]
    b.bar(x - w / 2, nm["I_bf"], w, color=BLUE, lw=0, label=f"single-reference reactions (n = {int((~is_mr).sum())})")
    b.bar(x + w / 2, mrr["I_bf"], w, color=ORANGE, lw=0, label=f"reactions with W4-17-MR species (n = {int(is_mr.sum())})")
    for i, v in enumerate(d["I_bf_gt1_pct"]):
        b.text(x[i], max(nm["I_bf"].iloc[i], mrr["I_bf"].iloc[i]) + 0.05, f"{v:.0f} %\n> 1 kcal/mol", ha="center", fontsize=7, color=MUTED)
    b.set_xticks(x); b.set_xticklabels(lab, fontsize=7.5); b.set_ylabel("MAE of the interaction I_bf (kcal/mol)"); b.set_ylim(0, max(1.75, d["I_bf"].max() * 1.5))
    b.set_title("(b) Non-additivity depends on the rung, not on MR character"); b.legend(frameon=False, fontsize=7.5, loc="upper left")
    b.grid(axis="y", color="#eeede9", lw=0.6); b.set_axisbelow(True)
    n_pts = int(len(dft))
    fig.suptitle(f"ConformalLadder on W4-17 — {len(avail)} functionals × {{{B_C}, {B_BIG}}} + D3/D4 + FOD on 215 species\n({n_pts} DFT single points; W4 references)", fontsize=10.5, fontweight="semibold")
    save_fig(fig, RES / "w4_17_rung_comparison.png"); print("saved", RES / "w4_17_rung_comparison.png")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {CL_ROOT}/notebooks/make_tables.py
"""Collect every headline number into results/tables/ (CSV per table + SUMMARY.md).

Run after the pilots, gates and W4-17 scripts:  PYTHONPATH=src python notebooks/make_tables.py
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
R = ROOT / "results"; T = R / "tables"; T.mkdir(parents=True, exist_ok=True)
md = ["# ConformalLadder — Stage 1 summary tables\n"]


def emit(name: str, df: pd.DataFrame, title: str, note: str = ""):
    df.to_csv(T / f"{name}.csv", index=False)
    md.append(f"## {title}\n")
    if note:
        md.append(note + "\n")
    md.append(df.to_markdown(index=False, floatfmt=".3f") + "\n")


# ---------------- pilot 0 ----------------
p = R / "pilot0" / "pilot0_results.json"
if p.exists():
    r = json.load(open(p)); rows = []
    for name, v in r.items():
        e = v["layers"]["eps_total"] if "layers" in v else v
        rows.append({"scenario": name, "n_train": v["n_train"], "n_cal": v["n_cal"], "n_test": v["n_test"],
                     "MAE_raw": v["raw_error"]["MAE_B3LYP_vs_G4MP2"], "MAE_corrected": v["corrected_error"]["MAE"],
                     "coverage_split": v["scaled_split_conformal"]["coverage"], "coverage_CQR": v["CQR"]["coverage"],
                     "weighted_abstain": v["weighted_split_conformal"]["frac_abstained"],
                     "coverage_weighted_answered": v["weighted_split_conformal"]["coverage_among_answered"],
                     "BH_certified": v["certification_BH_with_abstention"]["fraction_selected"], "BH_FDP": v["certification_BH_with_abstention"]["FDP"],
                     "naive_certified": v["certification_naive_plugin"]["fraction_selected"], "naive_FDP": v["certification_naive_plugin"]["FDP"]})
    emit("T1_pilot0_qm9_g4mp2", pd.DataFrame(rows), "Table 1 — Pilot 0: B3LYP/6-31G(2df,p) → G4MP2 on QM9 (130 258 molecules), one lumped layer",
         "90 % intervals (α = 0.10); certification = |corrected error| < 1 kcal/mol at FDR q = 0.10.")
    emit("T1b_pilot0_fdp_curve", pd.DataFrame(r["A_random"]["fdp_curve"])[["q", "fraction_selected", "FDP", "power"]], "Table 1b — Pilot 0: empirical FDP versus nominal q (exchangeable split)")

# ---------------- pilot 1 ----------------
p = R / "pilot1_multixc" / "pilot1_results.json"
if p.exists():
    r = json.load(open(p)); rows = []
    for name, v in r["splits"].items():
        row = {"split": name, "n_train": v["n_train"], "n_cal": v["n_cal"], "n_test": v["n_test"]}
        for lay, d in v["layers"].items():
            row[f"{lay}_MAE_raw"] = d["MAE_raw"]; row[f"{lay}_MAE_model"] = d["MAE_after_model"]; row[f"{lay}_coverage"] = d["coverage"]
        e = v["layers"]["eps_total"]
        row.update({"BH_certified": e["certification_BH"]["fraction_selected"], "BH_FDP": e["certification_BH"]["FDP"],
                    "BH_gate_certified": e["certification_BH_with_abstention"]["fraction_selected"], "naive_certified": e["certification_naive_plugin"]["fraction_selected"], "naive_FDP": e["certification_naive_plugin"]["FDP"]})
        rows.append(row)
    emit("T2_pilot1_multixc_isomers", pd.DataFrame(rows), f"Table 2 — Pilot 1: four-layer ladder on MultiXC-QM9 isomerization energies ({r['n_pairs']} pairs; {r['cheap'][0]}/{r['cheap'][1]} → {r['big_basis']} → {r['best_functional']} → -D → G4MP2)")
    s = pd.read_csv(R / "pilot1_multixc" / "budget_summary_A.csv")
    emit("T2b_pilot1_budget_by_size", s[["subset", "n", "eps_total_MAE", "beta_basis_MAE", "phi_functional_MAE", "delta_dispersion_MAE", "rho_remainder_MAE", "I_bf_MAE"]],
         "Table 2b — Pilot 1: measured layer budget by molecule size (MAE, kcal/mol)")

# ---------------- pilot 2 ----------------
p = R / "pilot2_qm7b" / "pilot2_results.json"
if p.exists():
    r = json.load(open(p)); rows = []
    for prob, pr in r["problems"].items():
        for name, v in pr["splits"].items():
            row = {"problem": prob, "split": name, "n_train": v["n_train"], "n_cal": v["n_cal"], "n_test": v["n_test"]}
            for tg, d in v["targets"].items():
                row[f"{tg}_MAE_raw"] = d["MAE_raw"]; row[f"{tg}_MAE_model"] = d["MAE_after_model"]; row[f"{tg}_coverage"] = d["coverage"]
            i = v["targets"]["I_bc"]
            row.update({"composite_within1_raw": i["frac_raw_composite_within_1kcal"], "composite_within1_learned": i["frac_learned_composite_within_1kcal"],
                        "abstain": i["weighted_conformal"]["frac_abstained"],
                        "cert_gate_frac": i["certify_learned_composite"]["fraction_selected"], "cert_gate_FDP": i["certify_learned_composite"]["FDP"],
                        "cert_nogate_frac": i["certify_learned_composite_no_gate"]["fraction_selected"], "cert_nogate_FDP": i["certify_learned_composite_no_gate"]["FDP"],
                        "naive_frac": i["naive_rule"]["fraction_selected"], "naive_FDP": i["naive_rule"]["FDP"]})
            rows.append(row)
    emit("T3_pilot2_qm7b_additivity", pd.DataFrame(rows), "Table 3 — Pilot 2: correlation × basis non-additivity on the QM7b grid (MP2/6-31G → MP2/cc-pVDZ → CCSD(T)/cc-pVDZ)",
         "I_bc = [CCSD(T)−MP2]_cc-pVDZ − [CCSD(T)−MP2]_6-31G is the exact error of the additive composite scheme.")
    summ = [{"problem": k, **{f"{tg}_{m}": vv for tg, d in v["summary_kcal"].items() for m, vv in d.items()}} for k, v in r["problems"].items()]
    emit("T3b_pilot2_raw_layers", pd.DataFrame(summ), "Table 3b — Pilot 2: raw size of each layer (kcal/mol) before any learning")

# ---------------- gates / recalibration ----------------
p = R / "gates" / "gate_comparison.csv"
if p.exists():
    emit("T4_gate_comparison", pd.read_csv(p), "Table 4 — Abstention gates under distribution shift (certificate at q = 0.10)")
p = R / "gates" / "recalibration_summary.csv"
if p.exists():
    emit("T5_recalibration", pd.read_csv(p), "Table 5 — Escalate-to-recalibrate: coverage / width / FDP versus the number of escalated items")

# ---------------- W4-17 ----------------
p = R / "w4-17" / "rung_comparison.csv"
if p.exists():
    t = pd.read_csv(p)
    emit("T6_w4_17_rungs", t[t.subset == "ALL"].drop(columns=["subset"]), "Table 6 — W4-17: layer budget for each choice of the DFT rung (MAE, kcal/mol; cheap level PBE/def2-SVP; W4 references)")
    emit("T6b_w4_17_rungs_by_MR", t[t.subset != "ALL"].pivot(index="rung", columns="subset", values=["I_bf", "rho_after_ladder"]).reset_index(),
         "Table 6b — W4-17: non-additivity and remainder by multireference status")
p = R / "w4-17" / "results_PBE_def2-SVP.json"
if p.exists():
    r = json.load(open(p)); rows = []
    rep = r.get("repeats", {}).get("layers", {})
    for lay, d in r["layers"].items():
        row = {"layer": lay, "MAE_raw": d["MAE_raw"], "MAE_model": d["MAE_after_model"], "coverage": d["coverage"], "mean_width": d["mean_width"],
               "MR_OOD_coverage": d.get("OOD_multireference", {}).get("coverage", np.nan)}
        if lay in rep:   # mean ± s.d. over the repeated random splits
            row.update({"MAE_model_mean": rep[lay]["MAE_after_model"]["mean"], "MAE_model_sd": rep[lay]["MAE_after_model"]["sd"],
                        "coverage_mean": rep[lay]["coverage"]["mean"], "coverage_sd": rep[lay]["coverage"]["sd"],
                        "MR_OOD_coverage_mean": rep[lay].get("OOD_coverage", {}).get("mean", np.nan)})
        rows.append(row)
    emit("T7_w4_17_conformal_layers", pd.DataFrame(rows), f"Table 7 — W4-17: learned layers with conformal intervals (rung {r['best_functional']}-{r.get('disp_tag','')}; n_train {r['n_train']}, n_cal {r['n_cal']}, n_test {r['n_test']}, MR reactions {r['n_mr']}; primary split, and mean ± s.d. over {r.get('repeats', {}).get('n_splits', 1)} random splits)")
p = R / "w4-17" / "diagnostics.json"
if p.exists():
    emit("T8_w4_17_diagnostics", pd.DataFrame([json.load(open(p))]), "Table 8 — W4-17: cheap diagnostics")

(T / "SUMMARY.md").write_text("\n".join(md))
print("\n".join(md)[:6000])
print(f"\n... tables written to {T}")


In [ ]:
%%writefile {CL_ROOT}/notebooks/fig_schematic.py
"""Figure 1 — overview schematic of ConformalLadder (layers, guarantees, router)."""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from conformalladder.palette import AQUA, BLUE, FACE, GRAY, INK, MUTED, ORANGE, VERM, YELLOW, save_fig

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "results" / "figures" / "fig1_schematic.png"

W, H = 14.0, 6.0
FIGSIZE = (11.0, 4.72)
fig, ax = plt.subplots(figsize=FIGSIZE, facecolor=FACE)
ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")
fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)


def box(x, y, w, h, text, fc="white", ec=INK, lw=1.2, fs=6.9, weight="normal", color=INK, r=0.12):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}", fc=fc, ec=ec, lw=lw))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, fontweight=weight, color=color, linespacing=1.25)


def arrow(x1, y1, x2, y2, color=INK, lw=1.3, style="-|>", ls="-", shrink=2, rad=0.0):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle=style, mutation_scale=10, color=color, lw=lw, linestyle=ls,
                                 shrinkA=shrink, shrinkB=shrink, connectionstyle=f"arc3,rad={rad}"))


LG = "#F2F2F2"
# ---------------------------------------------------------------- (a) inputs (x 0.2 – 3.0)
ax.text(0.2, 5.65, "(a) one cheap calculation", fontsize=8.0, fontweight="bold", color=INK)
box(0.2, 4.15, 2.8, 1.15, "molecule or reaction computed\nonce at the cheap level\nc = (functional$_c$, basis$_c$)", fc=LG, ec=GRAY)
arrow(1.6, 4.15, 1.6, 3.75)
box(0.2, 2.35, 2.8, 1.4, "cheap diagnostics  x\nHOMO–LUMO gap, ⟨S$^2$⟩, FOD,\nspread over functionals,\ncomposition, bond counts", fc=LG, ec=GRAY)
arrow(1.6, 2.35, 1.6, 1.95)
box(0.2, 0.55, 2.8, 1.4, "learned layer models\nμ$_t$(x), σ$_t$(x) for every layer t\n(gradient boosting, training split)", fc="white", ec=BLUE)

# ---------------------------------------------------------------- (b) the ladder (x 3.5 – 9.65)
ax.text(3.5, 5.65, "(b) the ladder: an exact, layer-resolved error budget", fontsize=8.0, fontweight="bold", color=INK)
rungs = [("cheap functional / small basis   (level c)", LG, GRAY),          # bottom
         ("cheap functional / large basis", "white", BLUE),
         ("best functional / large basis", "white", ORANGE),
         ("best functional + dispersion / large basis", "white", AQUA),
         ("reference  (W4 / CCSD(T) / G4MP2)", LG, GRAY)]                  # top
x_l, w_l = 3.5, 3.2
y0, dh, bh = 0.85, 0.95, 0.6
for i, (t, fc, ec) in enumerate(rungs):
    y = y0 + i * dh
    box(x_l, y, w_l, bh, t, fc=fc, ec=ec, fs=6.6)
    if i < 4:
        arrow(x_l + w_l / 2, y + bh, x_l + w_l / 2, y + dh, color=INK, lw=1.0)
layers = [("β   basis", BLUE), ("φ   functional", ORANGE), ("δ   dispersion", AQUA), ("ρ   remainder", YELLOW)]
lab_y = [y0 + i * dh + bh + (dh - bh) / 2 for i in range(4)]
for (t, c), y in zip(layers, lab_y):
    ax.text(x_l + w_l + 0.15, y, t, fontsize=7.2, color=c, va="center", fontweight="bold")
# I_bf bracket spans the basis and functional layers
xb = x_l + w_l + 1.25
ax.annotate("", xy=(xb, lab_y[0]), xytext=(xb, lab_y[1]), arrowprops=dict(arrowstyle="<->", color=INK, lw=0.9))
ax.text(xb + 0.06, 0.5 * (lab_y[0] + lab_y[1]), "I$_{bf}$", fontsize=6.8, color=INK, va="center", ha="left")
box(8.1, 4.4, 1.55, 0.9, "ε = β + φ + δ + ρ\nexact identity;\nI$_{bf}$ = non-additivity", fc="white", ec=INK, fs=6.3)
box(8.1, 2.75, 1.55, 1.35, "every layer and the\ntotal get a distribution-\nfree interval\nμ$_t$(x) ± q̂$_t$σ$_t$(x)\ncoverage ≥ 1 − α", fc="white", ec=BLUE, fs=6.3)
arrow(3.0, 1.25, x_l, y0 + bh / 2, color=BLUE)
# learned models -> intervals: dotted route under the ladder
ax.plot([1.6, 1.6, 8.85], [0.55, 0.3, 0.3], color=BLUE, lw=1.0, ls=":")
arrow(8.85, 0.3, 8.85, 2.75, color=BLUE, ls=":", lw=1.0, shrink=0)

# ---------------------------------------------------------------- (c) router (x 10.1 – 13.6)
ax.text(10.1, 5.65, "(c) router with guarantees", fontsize=8.0, fontweight="bold", color=INK)
box(10.1, 4.45, 3.5, 0.85, "exchangeability test:\nis the new batch separable from the calibration set?", fc="white", ec=INK, fs=6.6)
box(10.1, 2.95, 1.6, 1.0, "conformal p-values\n+ Benjamini–Hochberg\n(FDR ≤ q)", fc="white", ec=AQUA, fs=6.4)
box(12.0, 2.95, 1.6, 1.0, "abstain:\nescalate ≥ 50 items,\nrecalibrate", fc="white", ec=VERM, fs=6.4)
arrow(10.9, 4.45, 10.9, 3.95, color=AQUA); ax.text(10.75, 4.2, "no", fontsize=6.8, color=AQUA, ha="right", va="center")
arrow(12.8, 4.45, 12.8, 3.95, color=VERM); ax.text(12.95, 4.2, "yes", fontsize=6.8, color=VERM, va="center")
box(10.1, 1.35, 1.6, 1.0, "certified:\nkeep the cheap\nresult", fc="white", ec=AQUA, fs=6.6)
box(12.0, 1.35, 1.6, 1.0, "not certified:\nescalate one rung", fc="white", ec=ORANGE, fs=6.6)
arrow(10.6, 2.95, 10.6, 2.35, color=AQUA); arrow(11.4, 2.95, 12.3, 2.35, color=ORANGE)
arrow(13.6, 3.55, 13.6, 4.75, color=VERM, lw=1.2, rad=-0.5)
ax.text(12.8, 0.85, "loop until the batch\nis exchangeable", fontsize=6.2, color=VERM, ha="center")
ax.text(10.9, 0.85, "certificate: |error| < 1 kcal/mol\nwith false-discovery rate ≤ q", fontsize=6.2, color=MUTED, ha="center")
arrow(9.65, 3.45, 10.1, 3.45, color=BLUE, ls=":", lw=1.0)

save_fig(fig, OUT); print("saved", OUT)


## 3. Data
Downloads QM9-G4MP2 (GitHub), ACCDB (GitHub), MultiXC-QM9 `molecules_qm9.csv` (DTU Data / Figshare, 533 MB) and the QM7b multilevel SI
(direct request, then attached datasets under `/kaggle/input`). Files already present are skipped, so the cell is safe to re-run.

In [ ]:
run("python -m conformalladder.kaggle_data --root " + str(ROOT), threads=NT, tail=40)
if CLEAN_RAW_CSV:
    csv = ROOT / "data/raw/multixc_qm9/molecules_qm9.csv"
    if csv.exists() and (ROOT / "data/processed/multixc_all_bases.parquet").exists():
        csv.unlink(); print("removed raw csv (parquet kept)")
AVAIL = json.loads((ROOT / "data/processed/AVAILABLE.json").read_text())
HAVE_QM7B = (ROOT / "data/raw/qm7b_multilevel/supplementary/Etot_ccpvdz.txt").exists()
for p in ["data/raw/g4mp2-atomization-energy/data/output/g4mp2_data.json.gz", "data/processed/multixc_all_bases.parquet",
          "data/raw/ACCDB/Geometries", "data/raw/qm7b_multilevel/supplementary/Etot_ccpvdz.txt", "data/processed/w4-17_grid.jsonl"]:
    print(("OK      " if (ROOT / p).exists() else "MISSING ") + p)
if not HAVE_QM7B:
    print("\nQM7b is missing -> pilot 2 and its shift scenarios will be skipped. To include it, attach ct8b00832_si_001.zip "
          "(Kaggle extracts it automatically; the extracted 'supplementary' folder is found as well).")

## 4. Pilot 0 — QM9-G4MP2: one lumped rung, coverage, FDR, shift
Builds the tidy table (RDKit bond counts, ~1 min), then runs the four scenarios (~3 min).

In [ ]:
run("python src/conformalladder/data_qm9g4mp2.py", threads=1, tail=12)
run("python notebooks/pilot0_qm9_g4mp2.py", threads=NT, tail=5)
run("python notebooks/pilot0_figure.py", threads=1, tail=2)
show("results/pilot0/pilot0_summary.png")

## 5. Pilot 1 — MultiXC-QM9 isomerization energies: four-layer ladder
Merges MultiXC-QM9 with QM9-G4MP2 (129 874 molecules), builds 21 468 formula-grouped isomer pairs, fits every layer with conformal intervals (~6 min).

In [ ]:
run("python src/conformalladder/data_multixc.py", threads=1, tail=6)
run("python notebooks/pilot1_multixc_isomers.py", threads=NT, tail=45)
run("python notebooks/pilot1_figure.py", threads=1, tail=2)
show("results/pilot1_multixc/pilot1_summary.png")

## 6. Pilot 2 — QM7b: is the composite (additivity) assumption safe?
Builds true atomization energies at all nine levels and measures I_bc = [CCSD(T)−MP2]_cc-pVDZ − [CCSD(T)−MP2]_6-31G per molecule (~3 min).

In [ ]:
if HAVE_QM7B:
    run("python src/conformalladder/data_qm7b.py", threads=1, tail=6)
    run("python notebooks/pilot2_qm7b_additivity.py", threads=NT, tail=45)
    run("python notebooks/pilot2_figure.py", threads=1, tail=2)
    show("results/pilot2_qm7b/pilot2_summary.png")
else:
    print("pilot 2 skipped: QM7b multilevel data not available")

## 7. Routing under shift — abstention gates and escalate-to-recalibrate
Gate comparison (~2 min) and the recalibration experiment (~5–8 min).

In [ ]:
run("python notebooks/gate_comparison.py", threads=NT, tail=60)
if RUN_RECALIBRATION:
    run("python notebooks/recalibration.py", threads=NT, tail=40)
    run("python notebooks/gates_figure.py", threads=1, tail=2)
    show("results/gates/gates_and_recalibration.png")

## 8. W4-17 — own functional × basis grid with PySCF
215 species × (8 functionals × {def2-SVP, def2-TZVP}) + D3(BJ)/D3(0)/D4 + FOD(TPSS, 5000 K), density fitting, 1 042 W4 relative energies.
The campaign appends to `data/processed/w4-17_grid.jsonl` and **resumes** if interrupted. It is skipped when a precomputed grid is attached.

In [ ]:
grid = ROOT / "data/processed/w4-17_grid.jsonl"
have = grid.exists() and "ALL DONE" in (grid.with_suffix(".log").read_text() if grid.with_suffix(".log").exists() else "")
if grid.exists() and not have:
    n = sum(1 for l in open(grid) if l.strip()); have = n >= 3600 + 5000   # full grid: 3655 DFT + 5160 dispersion records
if have:
    print("precomputed / complete grid found:", grid)
elif RUN_W4_CAMPAIGN:
    run("python -m conformalladder.compute_grid --set W4-17", threads=NT, tail=25)
else:
    print("campaign skipped (RUN_W4_CAMPAIGN = False)")
if grid.exists():
    import collections
    recs = [json.loads(l) for l in open(grid) if l.strip()]
    c = collections.Counter((r["xc"], r["basis"]) for r in recs if r["basis"] != "none")
    print({f"{k[0]}/{k[1]}": v for k, v in sorted(c.items())}, "| unconverged:", sum(1 for r in recs if r["basis"] != "none" and not r["converged"]))

In [ ]:
if grid.exists():
    run("python notebooks/w4_17_budget.py --cheap PBE def2-SVP --big def2-TZVP", threads=NT, tail=20)
    run("python notebooks/w4_17_figure.py PBE_def2-SVP", threads=1, tail=2)
    run("python notebooks/w4_17_rungs.py", threads=1, tail=15)
    show("results/w4-17/w4_17_budget_PBE_def2-SVP.png"); show("results/w4-17/w4_17_rung_comparison.png")

## 9. Tables and archive
All headline numbers as CSV + Markdown (`results/tables/`), then one zip with figures, tables, JSON results and the W4-17 grid.

In [ ]:
run("python notebooks/fig_schematic.py", threads=1, tail=2); show("results/figures/fig1_schematic.png")
run("python notebooks/make_tables.py", threads=1, tail=80)
import zipfile
out = ROOT.parent / "ConformalLadder_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        if p.is_file() and p.suffix in (".png", ".csv", ".md", ".json", ".jsonl", ".log") and "data/raw" not in str(p):
            z.write(p, p.relative_to(ROOT))
print(out, f"{out.stat().st_size/1e6:.1f} MB")
print((ROOT / "results/tables/SUMMARY.md").read_text()[:3000])